# Kazakh Legal RAG — Dataset-Based Evaluation Notebook

This revised notebook evaluates **Naive RAG, Dense RAG, Hybrid RAG, Static Full RAG, Oracle-Adaptive Retrieval, and true query-text-only Adaptive LegalRAG** on the fixed benchmark:

```python
BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
HF_CORPUS_ID  = 'Arailym-tleubayeva/LegalRAG'
OUTPUT_DIR    = Path('/content/legalrag_outputs')
```

Key fixes compared with the earlier reproduction notebook:

1. The benchmark is treated as a **242-row fixed dataset**, not as a 150-item balanced benchmark.
2. Retrieval metrics are computed on the **182 answerable** questions only.
3. Abstention diagnostics are computed on the **full 242-row benchmark**.
4. Adaptive LegalRAG no longer uses `gold complexity` during inference.
5. The old `row['complexity']` version is retained only as **Oracle Adaptive Retrieval** for ablation.
6. Retrieval evaluation now includes `DocRecall@10` and `SourceRecall@10`.
7. Reranking latency is recomputed fairly instead of timing only metric evaluation.
8. Text normalization is reported honestly as either corpus `normalized_text` or fallback **Kazakh character folding**.
9. Dense retriever selection is explicitly limited to the active model list unless optional models are enabled.

**This final version adds:**
10. Multi-gold chunk parsing fix in retrieval metrics.
11. Bootstrap 95% CI per system (per-question resampling) and cross-iteration reproducibility (mean + range).
12. Adjudicated human-eval aggregation, inter-annotator agreement readout, and qualitative QA example export.


---
# 0. Setup

In [1]:
#@title 0.1 Install dependencies
!pip install -q datasets rank-bm25 sentence-transformers faiss-cpu pandas numpy matplotlib openpyxl tqdm scikit-learn scipy openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 121.3 MB/s eta 0:00:00


In [2]:
#@title 0.2 Imports and global configuration
import os, re, json, time, shutil, zipfile, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
warnings.filterwarnings('ignore')

BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
HF_CORPUS_ID  = 'Arailym-tleubayeva/LegalRAG'
OUTPUT_DIR    = Path('/content/legalrag_outputs')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 256
RRF_K = 60
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Output dir: {OUTPUT_DIR}')


def norm_id(v):
    """Normalize ids read from CSV/HF datasets, especially float-like ids such as 123.0."""
    if pd.isna(v):
        return ''
    s = str(v).strip()
    if s.lower() in {'', 'nan', 'none', 'null'}:
        return ''
    return re.sub(r'\.0$', '', s)


def clean_str(v):
    if pd.isna(v):
        return ''
    return str(v).strip()


def save_table(df: pd.DataFrame, name: str):
    """Save a result table as CSV and XLSX under OUTPUT_DIR."""
    csv_path = OUTPUT_DIR / f'{name}.csv'
    xlsx_path = OUTPUT_DIR / f'{name}.xlsx'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    df.to_excel(xlsx_path, index=False)
    print(f'Saved: {csv_path.name}, {xlsx_path.name}')


def save_json(obj, name: str):
    path = OUTPUT_DIR / f'{name}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f'Saved: {path.name}')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Output dir: /content/legalrag_outputs


In [3]:
#@title 0.3 clean_str with float/ID normalization (FIX for 788.0 vs "788" bug)
import math

def clean_str(x):
    """Normalize any value to a clean string ID.
    Critical fix: pandas reads numeric chunk IDs as float (788.0), while corpus
    chunk IDs are strings ('788'). Without normalization, gold==retrieved checks
    silently fail. This maps 788.0 / '788.0' / 788 / '788' all to '788'."""
    if x is None:
        return ''
    if isinstance(x, float):
        if math.isnan(x):
            return ''
        if x.is_integer():
            return str(int(x))
        return str(x).strip()
    if isinstance(x, int):
        return str(x)
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return ''
    if s.endswith('.0') and s[:-2].isdigit():
        s = s[:-2]
    return s

assert clean_str(788.0) == clean_str('788') == clean_str(788) == clean_str('788.0') == '788', 'clean_str fix failed'
print('clean_str fixed: 788.0 / "788.0" / 788 / "788" all ->', clean_str(788.0))

clean_str fixed: 788.0 / "788.0" / 788 / "788" all -> 788


In [4]:
#@title 0.4 Normalize gold ID columns at load time (root-cause fix)
ID_COLS = ['gold_chunk_id', 'gold_doc_id', 'gold_chunk_id_str', 'gold_doc_id_str',
           'gold_source', 'gold_source_str']

def _normalize_id_columns(df):
    for col in ID_COLS:
        if col in df.columns:
            df[col] = (df[col].astype('string')
                       .str.replace(r'\.0$', '', regex=True)
                       .fillna(''))
    return df

for _name in ['bench_all', 'bench_ans', 'bench_unans']:
    if _name in dir():
        globals()[_name] = _normalize_id_columns(globals()[_name].copy())
        print(f'Normalized ID columns in {_name} ({len(globals()[_name])} rows)')

for _name in ['bench_all', 'bench_ans', 'bench_unans']:
    if _name in dir():
        _df = globals()[_name]
        if 'gold_chunk_id_str' not in _df.columns and 'gold_chunk_id' in _df.columns:
            _df['gold_chunk_id_str'] = _df['gold_chunk_id'].map(clean_str)
        if 'gold_doc_id_str' not in _df.columns and 'gold_doc_id' in _df.columns:
            _df['gold_doc_id_str'] = _df['gold_doc_id'].map(clean_str)
        if 'gold_source_str' not in _df.columns and 'gold_source' in _df.columns:
            _df['gold_source_str'] = _df['gold_source'].map(clean_str)

print('Gold ID columns normalized and *_str helpers ensured.')

Gold ID columns normalized and *_str helpers ensured.


In [5]:
#@title 0.5 Verify the ID fix on one question (must show in pool: True)
_q = bench_ans.iloc[0]['question']
_cands = rrf([bm25_ret(bm25_text, _q, 20), dense_ret(faiss_index, query_embs_ans[0], 20)], k=20)
_gold = clean_str(bench_ans.iloc[0].get('gold_chunk_id_str', bench_ans.iloc[0].get('gold_chunk_id', '')))
_ids = [clean_str(x) for x in _cands['retrieved_ids']]
print('gold:', _gold)
print('first retrieved ids:', _ids[:5])
print('gold in pool:', _gold in _ids, '| position:', (_ids.index(_gold) if _gold in _ids else 'NOT FOUND'))
print('--> If "gold in pool: True", the ID type bug is fixed. Now re-run all retrieval cells.')

NameError: name 'bench_ans' is not defined

---
# 1. Load benchmark and verify dataset composition

In [6]:
#@title 1.1 Load `legalrag_qa_benchmark.csv`
from pathlib import Path

# Colab upload fallback
if not Path(BENCHMARK_CSV).exists():
    local_fallbacks = [
        '/mnt/data/legalrag_qa_benchmark(2).csv',
        '/mnt/data/legalrag_qa_benchmark.csv',
        'legalrag_qa_benchmark.csv',
    ]
    found = next((p for p in local_fallbacks if Path(p).exists()), None)
    if found:
        BENCHMARK_CSV = found
        print(f'Using local fallback: {BENCHMARK_CSV}')
    else:
        try:
            from google.colab import files as cf
            uploaded = cf.upload()
            for fname in uploaded:
                Path(fname).rename('/content/legalrag_qa_benchmark.csv')
            BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
        except Exception as e:
            raise FileNotFoundError('Benchmark CSV not found. Upload legalrag_qa_benchmark.csv.') from e

bench_all = pd.read_csv(BENCHMARK_CSV, encoding='utf-8-sig')
bench_all = bench_all.reset_index(drop=False).rename(columns={'index': 'row_id'})

required_cols = [
    'question_id', 'question', 'gold_answer', 'gold_evidence', 'gold_chunk_id',
    'gold_doc_id', 'gold_source', 'gold_article', 'gold_paragraph', 'language',
    'answerability', 'legal_domain', 'complexity', 'question_type'
]
missing = [c for c in required_cols if c not in bench_all.columns]
if missing:
    raise ValueError(f'Missing benchmark columns: {missing}')

for col in ['gold_chunk_id', 'gold_doc_id', 'gold_source', 'gold_article', 'gold_paragraph']:
    bench_all[f'{col}_str'] = bench_all[col].apply(norm_id if 'id' in col else clean_str)

bench_all['answerability'] = bench_all['answerability'].fillna('').astype(str).str.strip()
bench_all['complexity'] = bench_all['complexity'].fillna('').astype(str).str.strip()
bench_all['question_type'] = bench_all['question_type'].fillna('').astype(str).str.strip()
bench_all['legal_domain'] = bench_all['legal_domain'].fillna('').astype(str).str.strip()
bench_all['question'] = bench_all['question'].fillna('').astype(str)

bench_ans = bench_all[bench_all['answerability'].eq('answerable')].reset_index(drop=True)
bench_unans = bench_all[bench_all['answerability'].eq('unanswerable')].reset_index(drop=True)

print(f'Benchmark rows: {len(bench_all)}')
print(f'Answerable: {len(bench_ans)} | Unanswerable: {len(bench_unans)}')
print(f'Unique questions: {bench_all["question"].nunique()}')
print('Answerable complexity:', dict(bench_ans['complexity'].value_counts()))

Benchmark rows: 242
Answerable: 182 | Unanswerable: 60
Unique questions: 186
Answerable complexity: {'complex': np.int64(75), 'moderate': np.int64(65), 'simple': np.int64(42)}


In [7]:
#@title 1.2 Experiment 0 — Benchmark Sanity Analysis
sanity_rows = [
    ('Total questions', len(bench_all)),
    ('Answerable', int(bench_all['answerability'].eq('answerable').sum())),
    ('Unanswerable', int(bench_all['answerability'].eq('unanswerable').sum())),
    ('Simple answerable', int(bench_ans['complexity'].eq('simple').sum())),
    ('Moderate answerable', int(bench_ans['complexity'].eq('moderate').sum())),
    ('Complex answerable', int(bench_ans['complexity'].eq('complex').sum())),
    ('Deadline questions', int(bench_all['question_type'].eq('deadline').sum())),
    ('Definition questions', int(bench_all['question_type'].eq('definition').sum())),
    ('Fact lookup questions', int(bench_all['question_type'].eq('fact_lookup').sum())),
    ('Unique questions', int(bench_all['question'].nunique())),
    ('Article-annotated', int(bench_all['gold_article_str'].astype(bool).sum())),
    ('Paragraph-annotated', int(bench_all['gold_paragraph_str'].astype(bool).sum())),
    ('Language', ', '.join(sorted(bench_all['language'].dropna().astype(str).unique()))),
]
sanity_df = pd.DataFrame(sanity_rows, columns=['Statistic', 'Value'])
display(sanity_df)
save_table(sanity_df, 'table00_benchmark_sanity')

answerability_dist = bench_all['answerability'].value_counts(dropna=False).rename_axis('answerability').reset_index(name='count')
complexity_dist = bench_ans['complexity'].value_counts(dropna=False).rename_axis('complexity').reset_index(name='count')
qtype_dist = bench_all['question_type'].value_counts(dropna=False).rename_axis('question_type').reset_index(name='count')
domain_dist = bench_all['legal_domain'].value_counts(dropna=False).rename_axis('legal_domain').reset_index(name='count')

save_table(answerability_dist, 'table00_answerability_distribution')
save_table(complexity_dist, 'table00_answerable_complexity_distribution')
save_table(qtype_dist, 'table00_question_type_distribution')
save_table(domain_dist, 'table00_legal_domain_distribution')

LIMITATION_TEXT = (
    'The benchmark is dominated by deadline and definition questions. '
    'Multi-hop, procedure, eligibility, obligation, and exception-based question types are absent '
    'from the current benchmark and are reserved for future work.'
)
print('\nLimitation to report in the paper:')
print(LIMITATION_TEXT)

,Statistic,Value
0,Total questions,242
1,Answerable,182
2,Unanswerable,60
3,Simple answerable,42
4,Moderate answerable,65
5,Complex answerable,75
6,Deadline questions,134
7,Definition questions,45
8,Fact lookup questions,3
9,Unique questions,186


Saved: table00_benchmark_sanity.csv, table00_benchmark_sanity.xlsx
Saved: table00_answerability_distribution.csv, table00_answerability_distribution.xlsx
Saved: table00_answerable_complexity_distribution.csv, table00_answerable_complexity_distribution.xlsx
Saved: table00_question_type_distribution.csv, table00_question_type_distribution.xlsx
Saved: table00_legal_domain_distribution.csv, table00_legal_domain_distribution.xlsx

Limitation to report in the paper:
The benchmark is dominated by deadline and definition questions. Multi-hop, procedure, eligibility, obligation, and exception-based question types are absent from the current benchmark and are reserved for future work.


In [8]:
#@title 1.3 Dataset composition figures
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(5, 5))
answerability_dist.set_index('answerability')['count'].plot(kind='pie', autopct='%1.1f%%', ax=ax)
ax.set_ylabel('')
ax.set_title('Benchmark Answerability')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_answerability.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
complexity_dist.set_index('complexity')['count'].reindex(['simple', 'moderate', 'complex']).dropna().plot(kind='bar', ax=ax)
ax.set_title('Answerable Questions by Complexity')
ax.set_xlabel('Complexity')
ax.set_ylabel('Count')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_complexity.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
qtype_dist.set_index('question_type')['count'].plot(kind='bar', ax=ax)
ax.set_title('Question Type Distribution')
ax.set_xlabel('Question type')
ax.set_ylabel('Count')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_question_type.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
domain_dist.set_index('legal_domain')['count'].plot(kind='bar', ax=ax)
ax.set_title('Legal Domain Distribution')
ax.set_xlabel('Legal domain')
ax.set_ylabel('Count')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig00_legal_domain.png', dpi=180, bbox_inches='tight')
plt.show()

---
# 2. Load LegalRAG corpus from Hugging Face

In [9]:
#@title 2.1 Load `Arailym-tleubayeva/LegalRAG`
from datasets import load_dataset

ds = load_dataset(HF_CORPUS_ID)
split = 'train' if 'train' in ds else list(ds.keys())[0]
corpus_df = ds[split].to_pandas().reset_index(drop=True)

print(f'HF dataset: {HF_CORPUS_ID} | split={split}')
print(f'Corpus rows: {len(corpus_df):,}')
print('Columns:', corpus_df.columns.tolist())


def choose_col(candidates, required=True):
    for c in candidates:
        if c in corpus_df.columns:
            return c
    if required:
        raise ValueError(f'None of these corpus columns found: {candidates}')
    return None

CHUNK_COL = choose_col(['chunk_id', 'id', 'chunkId'])
DOC_COL = choose_col(['doc_id', 'document_id', 'docid', 'documentId'], required=False)
SOURCE_COL = choose_col(['source', 'url', 'source_url', 'link'], required=False)
TEXT_COL = choose_col(['text', 'content', 'chunk_text'])
NORM_TEXT_COL = choose_col(['normalized_text', 'text_normalized', 'norm_text'], required=False)
TITLE_COL = choose_col(['title', 'doc_title', 'document_title'], required=False)

corpus_df['chunk_id_str'] = corpus_df[CHUNK_COL].apply(norm_id)
corpus_df['doc_id_str'] = corpus_df[DOC_COL].apply(norm_id) if DOC_COL else ''
corpus_df['source_str'] = corpus_df[SOURCE_COL].apply(clean_str) if SOURCE_COL else ''
corpus_df['text_str'] = corpus_df[TEXT_COL].fillna('').astype(str)
corpus_df['title_str'] = corpus_df[TITLE_COL].fillna('').astype(str) if TITLE_COL else ''

c_ids = corpus_df['chunk_id_str'].tolist()
c_docs = corpus_df['doc_id_str'].tolist()
c_srcs = corpus_df['source_str'].tolist()
c_text = corpus_df['text_str'].tolist()
c_titles = corpus_df['title_str'].tolist()

id_to_pos = {cid: i for i, cid in enumerate(c_ids)}
id2text = dict(zip(c_ids, c_text))
id2doc = dict(zip(c_ids, c_docs))
id2src = dict(zip(c_ids, c_srcs))
id2title = dict(zip(c_ids, c_titles))

gold_chunks = set(bench_ans['gold_chunk_id_str'].dropna().astype(str)) - {''}
covered = gold_chunks & set(c_ids)
coverage = len(covered) / max(1, len(gold_chunks))
print(f'Gold chunk coverage: {len(covered)}/{len(gold_chunks)} ({coverage:.1%})')

corpus_stats = pd.DataFrame([
    ('HF corpus id', HF_CORPUS_ID),
    ('Split', split),
    ('Rows / chunks', len(corpus_df)),
    ('Chunk id column', CHUNK_COL),
    ('Document id column', DOC_COL or ''),
    ('Source column', SOURCE_COL or ''),
    ('Text column', TEXT_COL),
    ('Normalized text column', NORM_TEXT_COL or 'not available'),
    ('Gold chunk coverage', f'{len(covered)}/{len(gold_chunks)} ({coverage:.1%})'),
], columns=['Statistic', 'Value'])
display(corpus_stats)
save_table(corpus_stats, 'table00_corpus_loading_stats')

README.md:   0%|          | 0.00/5.74k [00:00<?, ?B/s]

chunked_all_docs_structured_fixed.json:   0%|          | 0.00/256M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/56916 [00:00<?, ? examples/s]

HF dataset: Arailym-tleubayeva/LegalRAG | split=train
Corpus rows: 56,916
Columns: ['doc_id', 'date', 'section_title', 'source', 'chunk_id', 'text', 'words', 'char_len', 'word_len', 'normalized_text']
Gold chunk coverage: 182/182 (100.0%)


,Statistic,Value
0,HF corpus id,Arailym-tleubayeva/LegalRAG
1,Split,train
2,Rows / chunks,56916
3,Chunk id column,chunk_id
4,Document id column,doc_id
5,Source column,source
6,Text column,text
7,Normalized text column,normalized_text
8,Gold chunk coverage,182/182 (100.0%)


Saved: table00_corpus_loading_stats.csv, table00_corpus_loading_stats.xlsx


---
# 3. Metrics and retrieval utilities

In [10]:
#@title 3.1 Retrieval result object and metrics

def make_result(indices, scores=None, score_name='score'):
    ids, docs, srcs, texts, titles, scs = [], [], [], [], [], []
    if scores is None:
        scores = [np.nan] * len(indices)
    for idx, score in zip(indices, scores):
        idx = int(idx)
        if idx < 0 or idx >= len(c_ids):
            continue
        ids.append(c_ids[idx])
        docs.append(c_docs[idx])
        srcs.append(c_srcs[idx])
        texts.append(c_text[idx])
        titles.append(c_titles[idx])
        scs.append(float(score) if score is not None and not pd.isna(score) else np.nan)
    return {
        'retrieved_ids': ids,
        'retrieved_docs': docs,
        'retrieved_sources': srcs,
        'retrieved_texts': texts,
        'retrieved_titles': titles,
        'retrieved_scores': scs,
        'score_name': score_name,
    }


def hit_at_1(ids, gold_set):
    return int(bool(ids) and ids[0] in gold_set)


def recall_at_k(ids, gold_set, k):
    if not gold_set:
        return np.nan
    return len(set(ids[:k]) & gold_set) / len(gold_set)


def mrr_at_k(ids, gold_set, k=10):
    for rank, cid in enumerate(ids[:k], 1):
        if cid in gold_set:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ids, gold_set, k=10):
    if not gold_set:
        return np.nan
    dcg = sum(1.0 / np.log2(rank + 1) for rank, cid in enumerate(ids[:k], 1) if cid in gold_set)
    ideal_hits = min(len(gold_set), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg else 0.0


def exact_value_recall_at_k(values, gold_value, k):
    gold_value = clean_str(gold_value)
    if not gold_value:
        return np.nan
    return int(gold_value in [clean_str(v) for v in values[:k]])


def eval_retrieval(results, bench, kset=(1, 3, 5, 10)):
    """Evaluate retrieval on answerable questions with gold_chunk_id/gold_doc_id/gold_source."""
    rows = []
    for res, (_, row) in zip(results, bench.iterrows()):
        # FIX: support one OR several gold chunk ids separated by | ; ,
        gold_chunk = {p for p in re.split(r'[|;,]\s*', clean_str(row.get('gold_chunk_id_str', ''))) if p} - {''}
        ids = [clean_str(x) for x in res.get('retrieved_ids', [])]
        docs = [clean_str(x) for x in res.get('retrieved_docs', [])]
        srcs = [clean_str(x) for x in res.get('retrieved_sources', [])]
        scores = res.get('retrieved_scores', [])
        out = {
            'question_id': row.get('question_id', ''),
            'row_id': row.get('row_id', ''),
            'complexity': row.get('complexity', ''),
            'question_type': row.get('question_type', ''),
            'legal_domain': row.get('legal_domain', ''),
            'Hit@1': hit_at_1(ids, gold_chunk),
            'MRR@10': mrr_at_k(ids, gold_chunk, 10),
            'nDCG@10': ndcg_at_k(ids, gold_chunk, 10),
            'DocRecall@10': exact_value_recall_at_k(docs, row.get('gold_doc_id_str', ''), 10),
            'SourceRecall@10': exact_value_recall_at_k(srcs, row.get('gold_source_str', ''), 10),
            'TopScore': float(scores[0]) if scores else np.nan,
            'TopChunk': ids[0] if ids else '',
            'GoldChunk': next(iter(gold_chunk)) if gold_chunk else '',
        }
        for k in kset:
            out[f'Recall@{k}'] = recall_at_k(ids, gold_chunk, k)
        rows.append(out)
    details = pd.DataFrame(rows)
    metric_cols = [
        'Hit@1', 'Recall@1', 'Recall@3', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10',
        'DocRecall@10', 'SourceRecall@10', 'TopScore'
    ]
    agg = {c: round(float(np.nanmean(details[c])), 4) for c in metric_cols if c in details.columns}
    return agg, details


def summarize_metric_table(rows, name, sort_col='Recall@10'):
    df = pd.DataFrame(rows)
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    display(df)
    save_table(df, name)
    return df

print('Metrics ready.')

Metrics ready.


In [11]:
#@title 3.2 Tokenization, BM25, dense retrieval, and RRF helpers
from rank_bm25 import BM25Okapi


def tok(text):
    return str(text).lower().split()


def build_bm25(texts, tokenizer=tok, desc='BM25 corpus'):
    tokenized = [tokenizer(t) for t in tqdm(texts, desc=desc)]
    return BM25Okapi(tokenized)


def bm25_ret(index, query, k=10, tokenizer=tok):
    scores = index.get_scores(tokenizer(query))
    top = np.argsort(scores)[::-1][:k]
    return make_result(top, scores[top], score_name='bm25')


def dense_ret(index, query_embedding, k=10):
    scores, idx = index.search(query_embedding.reshape(1, -1).astype('float32'), k)
    return make_result(idx[0], scores[0], score_name='dense_ip')


def rrf(result_list, k=10, rrf_k=RRF_K):
    scores = defaultdict(float)
    meta = {}
    for res in result_list:
        ids = res.get('retrieved_ids', [])
        for rank, cid in enumerate(ids, 1):
            scores[cid] += 1.0 / (rrf_k + rank)
            pos = ids.index(cid)
            meta[cid] = {
                'doc': res.get('retrieved_docs', [''])[pos] if pos < len(res.get('retrieved_docs', [])) else '',
                'source': res.get('retrieved_sources', [''])[pos] if pos < len(res.get('retrieved_sources', [])) else '',
                'text': res.get('retrieved_texts', [''])[pos] if pos < len(res.get('retrieved_texts', [])) else '',
                'title': res.get('retrieved_titles', [''])[pos] if pos < len(res.get('retrieved_titles', [])) else '',
            }
    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return {
        'retrieved_ids': top_ids,
        'retrieved_docs': [meta[cid]['doc'] for cid in top_ids],
        'retrieved_sources': [meta[cid]['source'] for cid in top_ids],
        'retrieved_texts': [meta[cid]['text'] for cid in top_ids],
        'retrieved_titles': [meta[cid]['title'] for cid in top_ids],
        'retrieved_scores': [scores[cid] for cid in top_ids],
        'score_name': 'rrf',
    }


def retrieval_confidence(res):
    scores = res.get('retrieved_scores', [])
    return float(scores[0]) if scores else 0.0

print('Retrieval utilities ready.')

Retrieval utilities ready.


---
# 4. Experiment 1 — Retrieval Baselines: BM25 vs Dense vs Hybrid

In [12]:
#@title 4.1 Build original BM25 index
print('Building BM25 on original corpus text...')
bm25_text = build_bm25(c_text, tok, desc='BM25-original')
print('BM25 ready.')

Building BM25 on original corpus text...


BM25-original:   0%|          | 0/56916 [00:00<?, ?it/s]

BM25 ready.


In [13]:
#@title 4.2 Dense retriever selection — sequential OOM-safe version

import gc, time, torch
from sentence_transformers import SentenceTransformer
import faiss

# Active models. For low-memory runs, temporarily leave only BGE-M3.
DENSE_MODELS = {
    'BGE-M3': 'BAAI/bge-m3',
    'E5-Large': 'intfloat/multilingual-e5-large',
    'LaBSE': 'sentence-transformers/LaBSE',
}

DENSE_BATCH_SIZE = 128      # 96 GB VRAM easily handles this at seq_len 512
MAX_SEQ_LENGTH = 512        # covers your longest chunks (~450 words), no truncation


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def model_prefixes(model_id):
    m = model_id.lower()
    if 'e5' in m:
        return 'passage: ', 'query: '
    return '', ''


def encode_texts_safe(model, texts, prefix='', batch_size=DENSE_BATCH_SIZE):
    prepared = [prefix + str(t) for t in texts]
    with torch.inference_mode():
        emb = model.encode(
            prepared,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=DEVICE,
        )
    return emb.astype('float32')


def build_faiss(embeddings):
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    return index


def evaluate_dense_model(name, model_id):
    """Load one model, evaluate it, then release GPU memory.
    This avoids keeping three large transformer models in memory at the same time.
    """
    print(f'\nLoading dense model: {name} — {model_id}')
    clear_gpu()

    model = SentenceTransformer(model_id, device=DEVICE, trust_remote_code=True)
    model.max_seq_length = MAX_SEQ_LENGTH

    corpus_prefix, query_prefix = model_prefixes(model_id)

    print('Encoding corpus...')
    corpus_emb = encode_texts_safe(model, c_text, corpus_prefix)
    faiss_index_tmp = build_faiss(corpus_emb)

    print('Encoding answerable questions...')
    q_emb_ans_tmp = encode_texts_safe(model, bench_ans['question'].tolist(), query_prefix)

    t0 = time.time()
    res = [dense_ret(faiss_index_tmp, q_emb_ans_tmp[i], k=10) for i in range(len(bench_ans))]
    latency = (time.time() - t0) / max(1, len(bench_ans)) * 1000

    metrics, details = eval_retrieval(res, bench_ans)
    metrics.update({
        'Model': name,
        'ModelId': model_id,
        'Latency_ms': round(latency, 2),
        'MaxSeqLength': MAX_SEQ_LENGTH,
        'BatchSize': DENSE_BATCH_SIZE,
    })

    print({k: metrics[k] for k in ['Hit@1', 'Recall@10', 'MRR@10', 'nDCG@10']})

    # Save only details and metrics; do not keep model/faiss_index for non-best models.
    save_table(details.assign(Model=name), f'detail01_dense_{name.replace("/", "_").replace(" ", "_")}')

    del model, corpus_emb, faiss_index_tmp, q_emb_ans_tmp, res
    clear_gpu()

    return metrics


dense_rows = []

for name, model_id in DENSE_MODELS.items():
    try:
        dense_rows.append(evaluate_dense_model(name, model_id))
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'CUDA OOM for {name}. Skipping this model. Try Runtime → Restart session or reduce MAX_SEQ_LENGTH.')
            clear_gpu()
        else:
            raise


dense_df = summarize_metric_table(dense_rows, 'table01_dense_retriever_selection')

if len(dense_df) == 0:
    raise RuntimeError('No dense model completed successfully. Use CPU runtime or keep only one model in DENSE_MODELS.')

BEST_DENSE = dense_df.sort_values(['Recall@10', 'MRR@10', 'nDCG@10'], ascending=False).iloc[0]['Model']
best_model_id = dense_df.loc[dense_df['Model'] == BEST_DENSE, 'ModelId'].iloc[0]
best_corpus_prefix, best_query_prefix = model_prefixes(best_model_id)

print(f'Best dense retriever by Recall@10/MRR/nDCG: {BEST_DENSE} — {best_model_id}')

# Reload only the best model for downstream experiments.
clear_gpu()
best_model = SentenceTransformer(best_model_id, device=DEVICE, trust_remote_code=True)
best_model.max_seq_length = MAX_SEQ_LENGTH

print('Re-encoding corpus with the best dense model for downstream experiments...')
best_corpus_emb = encode_texts_safe(best_model, c_text, best_corpus_prefix)
faiss_index = build_faiss(best_corpus_emb)

print('Encoding answerable questions with the best dense model...')
query_embs_ans = encode_texts_safe(best_model, bench_ans['question'].tolist(), best_query_prefix)

print('Encoding all 242 benchmark questions with the best dense model...')
query_embs_all = encode_texts_safe(best_model, bench_all['question'].tolist(), best_query_prefix)

del best_corpus_emb
clear_gpu()

print('Done.')



Loading dense model: BGE-M3 — BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Encoding answerable questions with the best dense model...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Encoding all 242 benchmark questions with the best dense model...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Done.


In [14]:
#@title 4.2b Add Qwen3 to the dense retriever comparison
# These two were missing because each needs special loading:
#   - Qwen3-Embedding-0.6B: requires a query instruction prefix for good retrieval
import gc, time
import numpy as np, pandas as pd
import torch
from sentence_transformers import SentenceTransformer

EXTRA_DENSE = {
    'Qwen3-0.6B': 'Qwen/Qwen3-Embedding-0.6B',
}

QWEN3_QUERY_INSTRUCTION = (
    'Instruct: Given a Kazakh legal question, retrieve relevant legal provisions\nQuery: '
)

def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def encode_corpus_and_eval(name, model_id):
    print(f'\n=== {name} ({model_id}) ===')
    is_jina = 'jina' in model_id.lower()
    is_qwen = 'qwen' in model_id.lower()

    st_kwargs = dict(device=DEVICE)
    if is_jina:
        st_kwargs['trust_remote_code'] = True
    model = SentenceTransformer(model_id, **st_kwargs)
    try:
        model.max_seq_length = MAX_SEQ_LENGTH
    except Exception:
        pass

    enc_kwargs = dict(batch_size=DENSE_BATCH_SIZE, show_progress_bar=True,
                      convert_to_numpy=True, normalize_embeddings=True)

    # corpus
    if is_jina:
        try:
            corpus_emb = model.encode(c_text, prompt_name='retrieval.passage', **enc_kwargs)
        except Exception:
            corpus_emb = model.encode(c_text, **enc_kwargs)
    else:
        corpus_emb = model.encode(c_text, **enc_kwargs)
    corpus_emb = corpus_emb.astype('float32')

    # queries
    questions = bench_ans['question'].tolist()
    if is_qwen:
        query_emb = model.encode([QWEN3_QUERY_INSTRUCTION + q for q in questions], **enc_kwargs)
    elif is_jina:
        try:
            query_emb = model.encode(questions, prompt_name='retrieval.query', **enc_kwargs)
        except Exception:
            query_emb = model.encode(questions, **enc_kwargs)
    else:
        query_emb = model.encode(questions, **enc_kwargs)
    query_emb = query_emb.astype('float32')

    import faiss
    index = faiss.IndexFlatIP(corpus_emb.shape[1])
    index.add(corpus_emb)

    start = time.time()
    results = [dense_ret(index, query_emb[i], k=10) for i in range(len(bench_ans))]
    lat = (time.time() - start) / len(bench_ans) * 1000

    metrics, details = eval_retrieval(results, bench_ans)
    metrics.update({'Model': name, 'ModelId': model_id, 'Latency_ms': round(lat, 2),
                    'MaxSeqLength': MAX_SEQ_LENGTH, 'BatchSize': DENSE_BATCH_SIZE})
    save_table(details, f'detail01_dense_{name}')
    print({k: metrics[k] for k in ['Hit@1','Recall@10','MRR@10','nDCG@10']})

    del model, corpus_emb, query_emb, index
    _free()
    return metrics

extra_rows = []
for name, mid in EXTRA_DENSE.items():
    try:
        extra_rows.append(encode_corpus_and_eval(name, mid))
    except Exception as e:
        print(f'FAILED {name}: {type(e).__name__}: {e}')

if extra_rows:
    new_df = pd.DataFrame(extra_rows)
    try:
        base = pd.read_csv(f'{OUTPUT_DIR}/table01_dense_retriever_selection.csv')
    except Exception:
        base = pd.DataFrame()
    combined = pd.concat([base, new_df], ignore_index=True)
    front = ['Model','Hit@1','Recall@1','Recall@3','Recall@5','Recall@10','MRR@10','nDCG@10','Latency_ms']
    cols = [c for c in front if c in combined.columns] + [c for c in combined.columns if c not in front]
    combined = combined[cols].sort_values('Recall@10', ascending=False).reset_index(drop=True)
    display(combined)
    save_table(combined, 'table01_dense_retriever_selection_full')
    print('Merged. Licenses for paper: Jina-v3 = CC-BY-NC-4.0 (non-commercial), Qwen3 = Apache-2.0.')
else:
    print('No extra models succeeded.')


=== Qwen3-0.6B (Qwen/Qwen3-Embedding-0.6B) ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/445 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Saved: detail01_dense_Qwen3-0.6B.csv, detail01_dense_Qwen3-0.6B.xlsx
{'Hit@1': 0.2473, 'Recall@10': 0.5055, 'MRR@10': 0.3211, 'nDCG@10': 0.3663}


,Model,Hit@1,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@10,Latency_ms,DocRecall@10,SourceRecall@10,TopScore,ModelId,MaxSeqLength,BatchSize
0,BGE-M3,0.4121,0.4121,0.5989,0.7033,0.7802,0.5324,0.5982,10.73,0.9176,0.9066,0.7099,BAAI/bge-m3,512,128
1,E5-Large,0.3297,0.3297,0.5055,0.5879,0.6703,0.4371,0.4947,10.62,0.8791,0.8571,0.8559,intfloat/multilingual-e5-large,512,128
2,Qwen3-0.6B,0.2473,0.2473,0.3516,0.4176,0.5055,0.3211,0.3663,10.68,0.8791,0.8791,0.7331,Qwen/Qwen3-Embedding-0.6B,512,128
3,LaBSE,0.0824,0.0824,0.1923,0.2473,0.3187,0.1471,0.1914,7.06,0.7527,0.7308,0.5396,sentence-transformers/LaBSE,512,128


Saved: table01_dense_retriever_selection_full.csv, table01_dense_retriever_selection_full.xlsx
Merged. Licenses for paper: Jina-v3 = CC-BY-NC-4.0 (non-commercial), Qwen3 = Apache-2.0.


In [15]:
#@title 4.3 BM25, Dense, and Hybrid comparison on 182 answerable questions
retrieval_rows = []
retrieval_details = {}

# Naive BM25 k=5
start = time.time()
bm25_k5 = [bm25_ret(bm25_text, row['question'], k=5, tokenizer=tok) for _, row in tqdm(bench_ans.iterrows(), total=len(bench_ans), desc='BM25 k=5')]
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(bm25_k5, bench_ans)
metrics.update({'System': 'Naive RAG: BM25 k=5', 'RetrievalType': 'lexical', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details['Naive RAG: BM25 k=5'] = details

# BM25 k=10
start = time.time()
bm25_k10 = [bm25_ret(bm25_text, row['question'], k=10, tokenizer=tok) for _, row in tqdm(bench_ans.iterrows(), total=len(bench_ans), desc='BM25 k=10')]
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(bm25_k10, bench_ans)
metrics.update({'System': 'BM25 k=10', 'RetrievalType': 'lexical', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details['BM25 k=10'] = details

# Dense best
start = time.time()
dense_k10 = [dense_ret(faiss_index, query_embs_ans[i], k=10) for i in tqdm(range(len(bench_ans)), desc=f'Dense {BEST_DENSE}')]
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(dense_k10, bench_ans)
metrics.update({'System': f'Dense {BEST_DENSE} k=10', 'RetrievalType': 'semantic', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details[f'Dense {BEST_DENSE} k=10'] = details

# Static Hybrid BM25 + Dense through RRF, k=10
start = time.time()
hybrid_k10 = []
for i, row in tqdm(list(bench_ans.iterrows()), desc='Hybrid k=10'):
    q = row['question']
    hybrid_k10.append(rrf([bm25_ret(bm25_text, q, k=10), dense_ret(faiss_index, query_embs_ans[i], k=10)], k=10))
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(hybrid_k10, bench_ans)
metrics.update({'System': 'Static Hybrid RAG: BM25+BGE-M3 RRF k=10', 'RetrievalType': 'hybrid', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details['Static Hybrid RAG'] = details

retrieval_df = summarize_metric_table(retrieval_rows, 'table02_retrieval_baselines')
save_table(pd.concat([d.assign(System=s) for s, d in retrieval_details.items()], ignore_index=True), 'detail02_retrieval_baselines')

# --- Save this run's retrieval summary tagged by ITERATION for cross-run reproducibility ---
ITERATION = int(os.environ.get('LEGALRAG_ITERATION', '1'))  # set 2 on the second run
retrieval_df.assign(iteration=ITERATION).to_csv(
    f'/content/retrieval_iter{ITERATION}.csv', index=False, encoding='utf-8-sig')
print(f'Saved /content/retrieval_iter{ITERATION}.csv  (set LEGALRAG_ITERATION=2 before the 2nd run)')


BM25 k=5:   0%|          | 0/182 [00:00<?, ?it/s]

BM25 k=10:   0%|          | 0/182 [00:00<?, ?it/s]

Dense BGE-M3:   0%|          | 0/182 [00:00<?, ?it/s]

Hybrid k=10:   0%|          | 0/182 [00:00<?, ?it/s]

,Hit@1,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@10,DocRecall@10,SourceRecall@10,TopScore,System,RetrievalType,Latency_ms
0,0.4505,0.4505,0.6923,0.7912,0.8571,0.5941,0.6584,0.9341,0.9286,0.0365,Static Hybrid RAG: BM25+BGE-M3 RRF k=10,hybrid,317.96
1,0.4176,0.4176,0.6484,0.7692,0.8352,0.5584,0.6278,0.9451,0.9286,39.2798,BM25 k=10,lexical,295.41
2,0.4121,0.4121,0.5989,0.7033,0.7802,0.5324,0.5982,0.9176,0.9066,0.7099,Dense BGE-M3 k=10,semantic,10.97
3,0.4176,0.4176,0.6484,0.7692,0.7692,0.5500,0.6068,0.9121,0.8956,39.2798,Naive RAG: BM25 k=5,lexical,302.63


Saved: table02_retrieval_baselines.csv, table02_retrieval_baselines.xlsx
Saved: detail02_retrieval_baselines.csv, detail02_retrieval_baselines.xlsx
Saved /content/retrieval_iter1.csv  (set LEGALRAG_ITERATION=2 before the 2nd run)


---
# 5. Experiment 2 — Original Text vs Normalized Text / Kazakh Folding

If the HF corpus contains `normalized_text`, this section uses it. Otherwise it reports the fallback as **Kazakh character folding**, not as full linguistic normalization.

In [16]:
#@title 5.1 Build normalized/folded BM25 index — safe version

KZ_FOLD = {
    'ә': 'а', 'і': 'и', 'ң': 'н', 'ғ': 'г',
    'ү': 'у', 'ұ': 'у', 'қ': 'к', 'ө': 'о', 'һ': 'х'
}


def fold_kazakh(text):
    s = str(text).lower()
    for a, b in KZ_FOLD.items():
        s = s.replace(a, b)
    return s


def tok_fold(text):
    return fold_kazakh(text).split()


def is_usable_text_column(texts, tokenizer, min_nonempty_ratio=0.05):
    nonempty = 0
    total = len(texts)

    for t in texts:
        try:
            if len(tokenizer(str(t))) > 0:
                nonempty += 1
        except Exception:
            pass

    ratio = nonempty / max(1, total)
    print(f'Non-empty tokenized docs: {nonempty}/{total} = {ratio:.4f}')
    return ratio >= min_nonempty_ratio


# 1) Try corpus normalized_text only if it is actually usable
use_corpus_normalized = False

if NORM_TEXT_COL and NORM_TEXT_COL in corpus_df.columns:
    candidate_texts = corpus_df[NORM_TEXT_COL].fillna('').astype(str).tolist()
    print(f'Found corpus normalized field: {NORM_TEXT_COL}')

    if is_usable_text_column(candidate_texts, tok):
        use_corpus_normalized = True
    else:
        print(f'Column {NORM_TEXT_COL} exists but is not usable for BM25. Falling back to Kazakh character folding.')
else:
    print('No corpus normalized_text column found. Falling back to Kazakh character folding.')


# 2) Select normalized/folded representation
if use_corpus_normalized:
    normalized_texts = candidate_texts
    normalized_label = 'BM25-normalized_text'
    normalized_tokenizer = tok

    def normalize_query_for_bm25(q):
        return str(q)

else:
    normalized_texts = [fold_kazakh(t) for t in c_text]
    normalized_label = 'BM25-fold Kazakh characters'
    normalized_tokenizer = tok

    def normalize_query_for_bm25(q):
        return fold_kazakh(q)


# 3) Final safety check before BM25
if not is_usable_text_column(normalized_texts, normalized_tokenizer, min_nonempty_ratio=0.05):
    raise ValueError(
        'Normalized/folded texts are still empty after fallback. '
        'Check c_text and corpus text column.'
    )


# 4) Build BM25
bm25_norm = build_bm25(
    normalized_texts,
    normalized_tokenizer,
    desc=normalized_label
)


# 5) Evaluate normalized/folded BM25
start = time.time()
bm25_norm_k10 = []

for _, row in tqdm(bench_ans.iterrows(), total=len(bench_ans), desc=normalized_label):
    q = normalize_query_for_bm25(row['question'])
    bm25_norm_k10.append(
        bm25_ret(
            bm25_norm,
            q,
            k=10,
            tokenizer=normalized_tokenizer
        )
    )

lat = (time.time() - start) / max(1, len(bench_ans)) * 1000


# 6) Compare original vs normalized/folded
m_orig, d_orig = eval_retrieval(bm25_k10, bench_ans)
m_norm, d_norm = eval_retrieval(bm25_norm_k10, bench_ans)

norm_rows = []

for system, metrics, latency in [
    ('BM25-original text', m_orig, np.nan),
    (normalized_label, m_norm, round(lat, 2)),
]:
    row = {
        'System': system,
        'Latency_ms': latency
    }

    for k in [
        'Hit@1', 'Recall@5', 'Recall@10',
        'MRR@10', 'nDCG@10',
        'DocRecall@10', 'SourceRecall@10'
    ]:
        if k in metrics:
            row[k] = metrics[k]

    norm_rows.append(row)

norm_df = pd.DataFrame(norm_rows)

metric_cols = [
    'Hit@1', 'Recall@5', 'Recall@10',
    'MRR@10', 'nDCG@10',
    'DocRecall@10', 'SourceRecall@10'
]

delta = {
    'System': f'Delta: {normalized_label} minus original',
    'Latency_ms': np.nan
}

for c in metric_cols:
    if c in norm_df.columns:
        delta[c] = round(float(norm_df.iloc[1][c] - norm_df.iloc[0][c]), 4)

norm_df = pd.concat([norm_df, pd.DataFrame([delta])], ignore_index=True)

display(norm_df)

save_table(norm_df, 'table03_text_normalization_or_folding')
save_table(
    d_norm.assign(System=normalized_label),
    'detail03_text_normalization_or_folding'
)

Found corpus normalized field: normalized_text
Non-empty tokenized docs: 0/56916 = 0.0000
Column normalized_text exists but is not usable for BM25. Falling back to Kazakh character folding.
Non-empty tokenized docs: 56916/56916 = 1.0000


BM25-fold Kazakh characters:   0%|          | 0/56916 [00:00<?, ?it/s]

BM25-fold Kazakh characters:   0%|          | 0/182 [00:00<?, ?it/s]

,System,Latency_ms,Hit@1,Recall@5,Recall@10,MRR@10,nDCG@10,DocRecall@10,SourceRecall@10
0,BM25-original text,NaN,0.4176,0.7692,0.8352,0.5584,0.6278,0.9451,0.9286
1,BM25-fold Kazakh characters,298.61,0.4286,0.7692,0.8352,0.5625,0.6306,0.9451,0.9286
2,Delta: BM25-fold Kazakh characters minus original,NaN,0.0110,0.0000,0.0000,0.0041,0.0028,0.0000,0.0000


Saved: table03_text_normalization_or_folding.csv, table03_text_normalization_or_folding.xlsx
Saved: detail03_text_normalization_or_folding.csv, detail03_text_normalization_or_folding.xlsx


---
# 6. Experiment 3 — Hybrid Retrieval and Reranking

The no-rerank baseline is recomputed end-to-end, so its latency is comparable with reranking variants.

In [19]:
#@title 3.2-fix rrf with correct text/doc alignment (no ids.index lookup)
def rrf(result_list, k=10, rrf_k=RRF_K):
    scores = defaultdict(float)
    meta = {}
    for res in result_list:
        ids    = res.get('retrieved_ids', [])
        docs   = res.get('retrieved_docs', [])
        srcs   = res.get('retrieved_sources', [])
        texts  = res.get('retrieved_texts', [])
        titles = res.get('retrieved_titles', [])
        for rank, cid in enumerate(ids):          # rank position = list index, synchronous
            scores[cid] += 1.0 / (rrf_k + rank + 1)
            # FIX: take metadata at THIS position (rank), not via ids.index(cid).
            # Only set if not already present, so the first retriever's aligned text wins.
            if cid not in meta:
                meta[cid] = {
                    'doc':    docs[rank]   if rank < len(docs)   else '',
                    'source': srcs[rank]   if rank < len(srcs)   else '',
                    'text':   texts[rank]  if rank < len(texts)  else '',
                    'title':  titles[rank] if rank < len(titles) else '',
                }
    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return {
        'retrieved_ids':     top_ids,
        'retrieved_docs':    [meta[cid]['doc']    for cid in top_ids],
        'retrieved_sources': [meta[cid]['source'] for cid in top_ids],
        'retrieved_texts':   [meta[cid]['text']   for cid in top_ids],
        'retrieved_titles':  [meta[cid]['title']  for cid in top_ids],
        'retrieved_scores':  [scores[cid]         for cid in top_ids],
        'score_name': 'rrf',
    }

In [20]:
#@title 6.1 Reranking ablation — FIXED (use texts/docs from RRF result, not id2* lookup)
from sentence_transformers import CrossEncoder

RUN_RERANKING = True
RERANKER_ID = 'BAAI/bge-reranker-v2-m3'
RERANK_POOLS = [20, 50, 100]

rerank_rows = []
rerank_details = {}

# Fair no-rerank baseline.
start = time.time()
hybrid_no_rerank = []
for i, row in tqdm(list(bench_ans.iterrows()), desc='Hybrid no rerank'):
    q = row['question']
    hybrid_no_rerank.append(rrf([bm25_ret(bm25_text, q, 10), dense_ret(faiss_index, query_embs_ans[i], 10)], k=10))
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(hybrid_no_rerank, bench_ans)
rerank_rows.append({'Pipeline': 'Hybrid only top-10', 'CandidatePool': 10, 'FinalTopK': 10, 'Latency_ms': round(lat, 2), **metrics})
rerank_details['Hybrid only top-10'] = details

if RUN_RERANKING:
    print(f'Loading reranker: {RERANKER_ID}')
    reranker = CrossEncoder(RERANKER_ID, device=DEVICE, max_length=512)

    def rerank_result(question, res, final_k=10):
        """Reorder candidates with a cross-encoder.
        FIX: take texts/docs/sources DIRECTLY from the RRF result (already aligned
        with retrieved_ids), instead of re-looking-up id2text/id2doc which can
        mismatch on key type or return truncated text. This keeps ids<->text<->doc
        perfectly synchronized, so reranked ids and their docs stay correct."""
        cids   = res.get('retrieved_ids', [])
        texts  = res.get('retrieved_texts', [])
        docs   = res.get('retrieved_docs', [])
        srcs   = res.get('retrieved_sources', [])
        titles = res.get('retrieved_titles', [])
        if not cids:
            return res
        # guard: align lengths (fall back to id2text only if RRF text missing)
        n = len(cids)
        def at(lst, i, fallback=''):
            return lst[i] if i < len(lst) and lst[i] not in (None, '') else fallback
        pair_texts = [at(texts, i) or id2text.get(cids[i], '') for i in range(n)]
        pairs = [(question, t) for t in pair_texts]
        scores = np.asarray(reranker.predict(pairs), dtype=float)
        order = np.argsort(scores)[::-1][:final_k]
        return {
            'retrieved_ids':     [cids[i] for i in order],
            'retrieved_docs':    [at(docs, i,   id2doc.get(cids[i], ''))   for i in order],
            'retrieved_sources': [at(srcs, i,   id2src.get(cids[i], ''))   for i in order],
            'retrieved_texts':   [pair_texts[i]                            for i in order],
            'retrieved_titles':  [at(titles, i, id2title.get(cids[i], '')) for i in order],
            'retrieved_scores':  [float(scores[i]) for i in order],
            'score_name': 'cross_encoder',
        }

    for pool in RERANK_POOLS:
        start = time.time()
        reranked = []
        for i, row in tqdm(list(bench_ans.iterrows()), desc=f'Hybrid -> reranker top-{pool}'):
            q = row['question']
            cands = rrf([bm25_ret(bm25_text, q, pool), dense_ret(faiss_index, query_embs_ans[i], pool)], k=pool)
            reranked.append(rerank_result(q, cands, final_k=10))
        lat = (time.time() - start) / len(bench_ans) * 1000
        metrics, details = eval_retrieval(reranked, bench_ans)
        rerank_rows.append({'Pipeline': f'Hybrid -> reranker top-{pool}', 'CandidatePool': pool, 'FinalTopK': 10, 'Latency_ms': round(lat, 2), **metrics})
        rerank_details[f'Hybrid -> reranker top-{pool}'] = details

rerank_df = pd.DataFrame(rerank_rows)
display(rerank_df)
save_table(rerank_df, 'table04_reranking_ablation')
save_table(pd.concat([d.assign(Pipeline=s) for s, d in rerank_details.items()], ignore_index=True), 'detail04_reranking_ablation')

if len(rerank_df) > 1:
    baseline_hit = float(rerank_df.iloc[0]['Hit@1'])
    best_rerank_hit = float(rerank_df.iloc[1:]['Hit@1'].max())
    print('Reranker improves Hit@1:', best_rerank_hit > baseline_hit,
          f'(baseline {baseline_hit:.3f} vs best rerank {best_rerank_hit:.3f})')


Hybrid no rerank:   0%|          | 0/182 [00:00<?, ?it/s]

Loading reranker: BAAI/bge-reranker-v2-m3


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Hybrid -> reranker top-20:   0%|          | 0/182 [00:00<?, ?it/s]

Hybrid -> reranker top-50:   0%|          | 0/182 [00:00<?, ?it/s]

Hybrid -> reranker top-100:   0%|          | 0/182 [00:00<?, ?it/s]

,Pipeline,CandidatePool,FinalTopK,Latency_ms,Hit@1,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@10,DocRecall@10,SourceRecall@10,TopScore
0,Hybrid only top-10,10,10,310.48,0.4505,0.4505,0.6923,0.7912,0.8571,0.5941,0.6584,0.9341,0.9286,0.0365
1,Hybrid -> reranker top-20,20,10,711.01,0.5604,0.5604,0.8187,0.8626,0.9121,0.6949,0.7486,0.9560,0.9505,0.9815
2,Hybrid -> reranker top-50,50,10,1244.84,0.5549,0.5549,0.8242,0.8516,0.9066,0.6913,0.7447,0.9725,0.9670,0.9858
3,Hybrid -> reranker top-100,100,10,2094.90,0.5549,0.5549,0.8187,0.8516,0.9121,0.6895,0.7443,0.9835,0.9835,0.9881


Saved: table04_reranking_ablation.csv, table04_reranking_ablation.xlsx
Saved: detail04_reranking_ablation.csv, detail04_reranking_ablation.xlsx
Reranker improves Hit@1: True (baseline 0.451 vs best rerank 0.560)


In [21]:
# Проверка типов в eval
print('gold_chunk_id dtype:', bench_ans['gold_chunk_id'].dtype)
print('gold sample:', bench_ans['gold_chunk_id'].head(3).tolist())
print('clean_str(788.0) =', repr(clean_str(788.0)))   # должно дать '788', а даёт?
print('clean_str("788") =', repr(clean_str("788")))

gold_chunk_id dtype: float64
gold sample: [788.0, 32.0, 413.0]
clean_str(788.0) = '788'
clean_str("788") = '788'


---
# 7. Query Analyzer — no gold-label leakage

Adaptive LegalRAG must receive **only the raw question text** at inference time. Gold labels such as `complexity`, `question_type`, `answerability`, `gold_answer`, `gold_evidence`, and `gold_chunk_id` are used only after inference for analysis.

In [22]:
#@title 7.1 Rule-based query analyzer using only question text
DEFINITION_MARKERS = [
    'дегеніміз не', 'анықтамасы', 'не болып табылады', 'кім болып табылады',
    'қандай ұғым', 'нені білдіреді', 'түсінігі'
]
DEADLINE_MARKERS = [
    'мерзім', 'неше күн', 'қанша күн', 'қанша уақыт', 'неше ай', 'қанша ай',
    'неше жыл', 'қанша жыл', 'күн ішінде', 'ай ішінде', 'жыл ішінде', 'дейін', 'кейін', 'бастап'
]
COMPLEX_MARKERS = [
    'қандай жағдайларда', 'қандай шарттар', 'қандай негіздер', 'қандай тәртіппен',
    'тәртібі', 'негіздері', 'ерекшеліктері', 'қоспағанда', 'егер', 'болған жағдайда',
    'бірнеше', 'қалай жүзеге асырылады', 'қалай белгіленеді'
]
UNANSWERABLE_RISK_MARKERS = [
    'болашақта', 'келесі жылы', '2026', '2027', 'болжам', 'қандай болады',
    'саяси', 'пікір', 'кеңес бер', 'адвокатсыз', 'сотта жеңемін бе'
]


def contains_any(text, markers):
    return any(m in text for m in markers)


def analyze_question(question: str):
    q = str(question).lower().strip()
    unans_risk = contains_any(q, UNANSWERABLE_RISK_MARKERS)

    if unans_risk:
        qtype = 'unanswerable_risk'
        complexity = 'complex'
        route = 'unanswerable_risk'
    elif contains_any(q, DEFINITION_MARKERS):
        qtype = 'definition'
        complexity = 'simple'
        route = 'definition_lookup'
    elif contains_any(q, DEADLINE_MARKERS):
        qtype = 'deadline'
        complexity = 'moderate'
        route = 'deadline_lookup'
    elif contains_any(q, COMPLEX_MARKERS):
        qtype = 'complex_legal'
        complexity = 'complex'
        route = 'complex_lookup'
    else:
        qtype = 'moderate_lookup'
        complexity = 'moderate'
        route = 'moderate_lookup'

    return {
        'pred_question_type': qtype,
        'pred_complexity': complexity,
        'pred_route': route,
        'pred_unanswerable_risk': int(unans_risk),
    }

qa_pred = pd.DataFrame([analyze_question(q) for q in bench_all['question']])
query_analyzer_df = pd.concat([bench_all[['row_id', 'question_id', 'question', 'answerability', 'complexity', 'question_type', 'legal_domain']], qa_pred], axis=1)
display(query_analyzer_df.head())
save_table(query_analyzer_df, 'detail05_query_analyzer_predictions')

,row_id,question_id,question,answerability,complexity,question_type,legal_domain,pred_question_type,pred_complexity,pred_route,pred_unanswerable_risk
0,0,TEST_0210,Жер учаскесін заңсыз пайдаланған жағдайда төле...,answerable,complex,deadline,tax_law,deadline,moderate,deadline_lookup,0
1,1,TEST_0056,KAZAKH INVEST агенттігі инвестиция тарту үшін ...,answerable,moderate,definition,migration_law,moderate_lookup,moderate,moderate_lookup,0
2,2,TEST_0235,Уранды барлауға арналған келісімшарт мерзімін ...,answerable,complex,deadline,civil_law,deadline,moderate,deadline_lookup,0
3,3,TEST_0227,Жер қойнауы учаскесін сенімгерлік басқару шарт...,answerable,complex,deadline,civil_law,deadline,moderate,deadline_lookup,0
4,4,TEST_0237,Жер қойнауын пайдалануға арналған келісімшартт...,answerable,complex,definition,civil_law,moderate_lookup,moderate,moderate_lookup,0


Saved: detail05_query_analyzer_predictions.csv, detail05_query_analyzer_predictions.xlsx


In [23]:
#@title 7.2 Post-hoc Query Analyzer evaluation against gold labels
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

qa_eval_rows = []
for gold_col, pred_col, task in [
    ('complexity', 'pred_complexity', 'Complexity prediction'),
    ('question_type', 'pred_question_type', 'Question type prediction'),
]:
    gold = query_analyzer_df[gold_col].astype(str)
    pred = query_analyzer_df[pred_col].astype(str)
    labels = sorted(set(gold) | set(pred))
    qa_eval_rows.append({
        'Task': task,
        'N': len(query_analyzer_df),
        'Accuracy': round(accuracy_score(gold, pred), 4),
        'MacroF1': round(f1_score(gold, pred, labels=labels, average='macro', zero_division=0), 4),
    })

# Unanswerable-risk detection: map answerability=unanswerable to positive class.
gold_unans = query_analyzer_df['answerability'].eq('unanswerable').astype(int)
pred_unans = query_analyzer_df['pred_unanswerable_risk'].astype(int)
qa_eval_rows.append({
    'Task': 'Unanswerable-risk detection',
    'N': len(query_analyzer_df),
    'Accuracy': round(accuracy_score(gold_unans, pred_unans), 4),
    'MacroF1': round(f1_score(gold_unans, pred_unans, average='macro', zero_division=0), 4),
})

qa_eval_df = pd.DataFrame(qa_eval_rows)
display(qa_eval_df)
save_table(qa_eval_df, 'table05_query_analyzer_posthoc_metrics')

route_dist = query_analyzer_df['pred_route'].value_counts().rename_axis('route').reset_index(name='count')
display(route_dist)
save_table(route_dist, 'table05_query_analyzer_route_distribution')

# Save crosstabs for error analysis.
ct_complexity = pd.crosstab(query_analyzer_df['complexity'], query_analyzer_df['pred_complexity'])
ct_qtype = pd.crosstab(query_analyzer_df['question_type'], query_analyzer_df['pred_question_type'])
ct_complexity.to_csv(OUTPUT_DIR / 'table05_crosstab_complexity.csv', encoding='utf-8-sig')
ct_qtype.to_csv(OUTPUT_DIR / 'table05_crosstab_question_type.csv', encoding='utf-8-sig')
print('Saved query analyzer crosstabs.')

,Task,N,Accuracy,MacroF1
0,Complexity prediction,242,0.4008,0.3011
1,Question type prediction,242,0.2107,0.0908
2,Unanswerable-risk detection,242,0.8430,0.7313


Saved: table05_query_analyzer_posthoc_metrics.csv, table05_query_analyzer_posthoc_metrics.xlsx


,route,count
0,moderate_lookup,146
1,deadline_lookup,54
2,unanswerable_risk,26
3,complex_lookup,13
4,definition_lookup,3


Saved: table05_query_analyzer_route_distribution.csv, table05_query_analyzer_route_distribution.xlsx
Saved query analyzer crosstabs.


---
# 8. Experiment 7 — Static Full RAG vs Adaptive LegalRAG retrieval-side evaluation

This section reports three variants:

1. **Static Hybrid RAG**: fixed BM25+BGE-M3 RRF retrieval for all questions.
2. **Oracle Adaptive Retrieval**: uses gold `complexity`; this is an ablation only and must not be claimed as the deployable Adaptive LegalRAG system.
3. **Adaptive LegalRAG**: uses `analyze_question(question)` and therefore receives only the raw question text.

In [24]:
#@title 8.1 Three-system retrieval policies — Naive / Static Hybrid / Adaptive (dynamic top-k, no rerank)
# Methodology (paper Section 3.6): adaptivity = COMPLEXITY-CONDITIONED DYNAMIC TOP-K.
#   simple  -> k=5  (compact evidence)
#   moderate-> k=10 (balanced)
#   complex -> k=15 (broad multi-provision)
# Fusion = BM25 + BGE-M3 via RRF + dedup. NO reranker (rerank is a separate ablation, Section 4.5).
# Oracle routing is NOT used in the paper; only Naive / Static / Adaptive are compared.

NAIVE_K = 5          # Naive RAG: BM25 only, fixed k
STATIC_K = 10        # Static Hybrid: BM25+BGE-M3, fixed k
ADAPTIVE_DEPTH = {'simple': 5, 'moderate': 10, 'complex': 15}
ADAPTIVE_DEFAULT_K = 10

def naive_retrieve(question, k=NAIVE_K):
    # Minimal non-adaptive baseline: sparse BM25 only.
    return bm25_ret(bm25_text, question, k=k, tokenizer=tok)

def static_hybrid_retrieve(question, query_embedding, k=STATIC_K):
    # Fixed-policy hybrid: same k for every question.
    return rrf([bm25_ret(bm25_text, question, k=k, tokenizer=tok),
                dense_ret(faiss_index, query_embedding, k=k)], k=k)

def adaptive_retrieve(question, query_embedding, depth_k):
    # Complexity-conditioned hybrid: dynamic k, same fusion as static.
    return rrf([bm25_ret(bm25_text, question, k=depth_k, tokenizer=tok),
                dense_ret(faiss_index, query_embedding, k=depth_k)], k=depth_k)

def run_naive():
    return [naive_retrieve(row['question'])
            for _, row in tqdm(list(bench_ans.iterrows()), desc='Naive RAG (BM25 k=5)')]

def run_static():
    return [static_hybrid_retrieve(row['question'], query_embs_ans[i])
            for i, row in tqdm(list(bench_ans.iterrows()), desc='Static Hybrid (k=10)')]

def run_adaptive():
    out, routes = [], []
    for i, row in tqdm(list(bench_ans.iterrows()), desc='Adaptive (dynamic top-k)'):
        pred = analyze_question(row['question'])
        # Map predicted complexity -> retrieval depth (paper 3.6.3 / Table 9).
        depth_k = ADAPTIVE_DEPTH.get(str(pred.get('pred_complexity', '')).lower(), ADAPTIVE_DEFAULT_K)
        out.append(adaptive_retrieve(row['question'], query_embs_ans[i], depth_k))
        routes.append({**pred, 'depth_k': depth_k})
    return out, pd.DataFrame(routes)

print('Stage-5 retrieval policies ready: Naive / Static Hybrid / Adaptive (dynamic top-k, no rerank).')


Stage-5 retrieval policies ready: Naive / Static Hybrid / Adaptive (dynamic top-k, no rerank).


In [25]:
#@title 8.2 Run the three systems on 182 answerable questions
start = time.time(); naive_res = run_naive(); naive_lat = (time.time()-start)/len(bench_ans)*1000
start = time.time(); static_res = run_static(); static_lat = (time.time()-start)/len(bench_ans)*1000
start = time.time(); adaptive_res, adaptive_routes = run_adaptive(); adaptive_lat = (time.time()-start)/len(bench_ans)*1000

systems = [
    ('Naive RAG', naive_res, naive_lat),
    ('Static Hybrid Legal RAG', static_res, static_lat),
    ('Adaptive Legal RAG', adaptive_res, adaptive_lat),
]

sys_rows, sys_details = [], []
for name, results, latency in systems:
    metrics, details = eval_retrieval(results, bench_ans)
    row = {'System': name, 'N': len(bench_ans), 'Latency_ms': round(latency, 2)}
    row.update(metrics)
    sys_rows.append(row)
    sys_details.append(details.assign(System=name))

three_sys_df = pd.DataFrame(sys_rows)
display(three_sys_df)
save_table(three_sys_df, 'table_three_systems_overall')
save_table(pd.concat(sys_details, ignore_index=True), 'detail_three_systems')
save_table(adaptive_routes, 'detail_adaptive_routes')

# keep a dict for downstream (significance, complexity breakdown)
three_sys_details = {name: det for (name, _, _), det in zip(systems, sys_details)}
print('Done. Naive/Static/Adaptive evaluated on', len(bench_ans), 'answerable questions.')


Naive RAG (BM25 k=5):   0%|          | 0/182 [00:00<?, ?it/s]

Static Hybrid (k=10):   0%|          | 0/182 [00:00<?, ?it/s]

Adaptive (dynamic top-k):   0%|          | 0/182 [00:00<?, ?it/s]

,System,N,Latency_ms,Hit@1,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@10,DocRecall@10,SourceRecall@10,TopScore
0,Naive RAG,182,289.89,0.4176,0.4176,0.6484,0.7692,0.7692,0.5500,0.6068,0.9121,0.8956,39.2798
1,Static Hybrid Legal RAG,182,309.25,0.4505,0.4505,0.6923,0.7912,0.8571,0.5941,0.6584,0.9341,0.9286,0.0365
2,Adaptive Legal RAG,182,310.31,0.4560,0.4560,0.6923,0.7912,0.8516,0.5972,0.6595,0.9341,0.9286,0.0365


Saved: table_three_systems_overall.csv, table_three_systems_overall.xlsx
Saved: detail_three_systems.csv, detail_three_systems.xlsx
Saved: detail_adaptive_routes.csv, detail_adaptive_routes.xlsx
Done. Naive/Static/Adaptive evaluated on 182 answerable questions.


In [26]:
#@title 8.3 Table 21 — Retrieval performance by query complexity (Static vs Adaptive)
# Single-gold benchmark: R@1 == Recall@1, so only R@1 is shown.
_static = three_sys_details['Static Hybrid Legal RAG']
_adapt  = three_sys_details['Adaptive Legal RAG']
COMPLEXITY_ORDER = ['simple', 'moderate', 'complex']

t21 = []
for comp in COMPLEXITY_ORDER:
    for label, det in [('Static Hybrid Legal RAG', _static), ('Adaptive Legal RAG', _adapt)]:
        sub = det[det['complexity'].astype(str).str.lower() == comp]
        t21.append({
            'Complexity Level': comp.capitalize(), 'System': label, 'n': len(sub),
            'R@1':       round(float(sub['Hit@1'].mean()), 3) if len(sub) else None,
            'Recall@10': round(float(sub['Recall@10'].mean()), 3) if len(sub) else None,
            'MRR':       round(float(sub['MRR@10'].mean()), 3) if len(sub) else None,
            'nDCG@10':   round(float(sub['nDCG@10'].mean()), 3) if len(sub) else None,
        })
table21 = pd.DataFrame(t21)
display(table21)
save_table(table21, 'table21_retrieval_by_complexity')
print('Table 21 ready. n per complexity among 182 answerable (simple 42 / moderate 65 / complex 75).')


,Complexity Level,System,n,R@1,Recall@10,MRR,nDCG@10
0,Simple,Static Hybrid Legal RAG,42,0.381,0.810,0.525,0.595
1,Simple,Adaptive Legal RAG,42,0.405,0.810,0.541,0.606
2,Moderate,Static Hybrid Legal RAG,65,0.431,0.831,0.576,0.639
3,Moderate,Adaptive Legal RAG,65,0.431,0.815,0.575,0.634
4,Complex,Static Hybrid Legal RAG,75,0.507,0.907,0.648,0.711
5,Complex,Adaptive Legal RAG,75,0.507,0.907,0.648,0.711


Saved: table21_retrieval_by_complexity.csv, table21_retrieval_by_complexity.xlsx
Table 21 ready. n per complexity among 182 answerable (simple 42 / moderate 65 / complex 75).


In [27]:
#@title 8.4 Significance testing — Wilcoxon signed-rank (paired) + Holm correction
from scipy.stats import wilcoxon
from itertools import combinations

SIG_METRICS = ['Hit@1', 'Recall@10', 'MRR@10', 'nDCG@10']
SYS_NAMES = ['Naive RAG', 'Static Hybrid Legal RAG', 'Adaptive Legal RAG']

# align per-question rows by row_id across systems
aligned = {name: three_sys_details[name].set_index('row_id') for name in SYS_NAMES}
common = set.intersection(*[set(d.index) for d in aligned.values()])
common = sorted(common)

sig_rows = []
for metric in SIG_METRICS:
    for a, b in combinations(SYS_NAMES, 2):
        va = aligned[a].loc[common, metric].astype(float).to_numpy()
        vb = aligned[b].loc[common, metric].astype(float).to_numpy()
        diff = va - vb
        if (diff == 0).all():
            stat, p = float('nan'), 1.0
        else:
            try:
                stat, p = wilcoxon(va, vb, zero_method='wilcox')
            except ValueError:
                stat, p = float('nan'), 1.0
        sig_rows.append({'Metric': metric, 'System A': a, 'System B': b,
                         'mean_A': round(float(va.mean()), 4),
                         'mean_B': round(float(vb.mean()), 4),
                         'p_raw': p})

sig_df = pd.DataFrame(sig_rows)
# Holm-Bonferroni within each metric family
sig_df['p_holm'] = float('nan')
for metric in SIG_METRICS:
    m = sig_df['Metric'] == metric
    order = sig_df[m].sort_values('p_raw').index
    n = len(order)
    prev = 0.0
    for rank, idx in enumerate(order):
        adj = min(1.0, (n - rank) * sig_df.loc[idx, 'p_raw'])
        adj = max(adj, prev)  # enforce monotonicity
        sig_df.loc[idx, 'p_holm'] = adj
        prev = adj
sig_df['sig_0.05'] = sig_df['p_holm'] < 0.05
sig_df['p_raw'] = sig_df['p_raw'].round(4)
sig_df['p_holm'] = sig_df['p_holm'].round(4)
display(sig_df)
save_table(sig_df, 'table_significance_wilcoxon')
print('Wilcoxon signed-rank (paired) on', len(common), 'common questions; Holm-corrected within each metric.')
print('sig_0.05=True means the paired difference is statistically significant after correction.')


,Metric,System A,System B,mean_A,mean_B,p_raw,p_holm,sig_0.05
0,Hit@1,Naive RAG,Static Hybrid Legal RAG,0.4176,0.4505,0.3545,0.8229,False
1,Hit@1,Naive RAG,Adaptive Legal RAG,0.4176,0.4560,0.2743,0.8229,False
2,Hit@1,Static Hybrid Legal RAG,Adaptive Legal RAG,0.4505,0.4560,0.3173,0.8229,False
3,Recall@10,Naive RAG,Static Hybrid Legal RAG,0.7692,0.8571,0.0001,0.0002,True
4,Recall@10,Naive RAG,Adaptive Legal RAG,0.7692,0.8516,0.0001,0.0002,True
5,Recall@10,Static Hybrid Legal RAG,Adaptive Legal RAG,0.8571,0.8516,0.3173,0.3173,False
6,MRR@10,Naive RAG,Static Hybrid Legal RAG,0.5500,0.5941,0.1005,0.2280,False
7,MRR@10,Naive RAG,Adaptive Legal RAG,0.5500,0.5972,0.0760,0.2280,False
8,MRR@10,Static Hybrid Legal RAG,Adaptive Legal RAG,0.5941,0.5972,1.0000,1.0000,False
9,nDCG@10,Naive RAG,Static Hybrid Legal RAG,0.6068,0.6584,0.0289,0.0675,False


Saved: table_significance_wilcoxon.csv, table_significance_wilcoxon.xlsx
Wilcoxon signed-rank (paired) on 182 common questions; Holm-corrected within each metric.
sig_0.05=True means the paired difference is statistically significant after correction.


In [29]:
#@title 8.3 Table 21 — Retrieval performance by query complexity (Static vs Adaptive)
_static = three_sys_details['Static Hybrid Legal RAG']
_adapt  = three_sys_details['Adaptive Legal RAG']
COMPLEXITY_ORDER = ['simple', 'moderate', 'complex']

t21 = []
for comp in COMPLEXITY_ORDER:
    for label, det in [('Static Hybrid Legal RAG', _static), ('Adaptive Legal RAG', _adapt)]:
        sub = det[det['complexity'].astype(str).str.lower() == comp]
        t21.append({
            'Complexity Level': comp.capitalize(), 'System': label, 'n': len(sub),
            'R@1':       round(float(sub['Hit@1'].mean()), 3) if len(sub) else None,
            'Recall@10': round(float(sub['Recall@10'].mean()), 3) if len(sub) else None,
            'MRR':       round(float(sub['MRR@10'].mean()), 3) if len(sub) else None,
            'nDCG@10':   round(float(sub['nDCG@10'].mean()), 3) if len(sub) else None,
        })
table21 = pd.DataFrame(t21)
display(table21)
save_table(table21, 'table21_retrieval_by_complexity')
print('Table 21 ready. n per complexity among 182 answerable.')

,Complexity Level,System,n,R@1,Recall@10,MRR,nDCG@10
0,Simple,Static Hybrid Legal RAG,42,0.381,0.810,0.525,0.595
1,Simple,Adaptive Legal RAG,42,0.405,0.810,0.541,0.606
2,Moderate,Static Hybrid Legal RAG,65,0.431,0.831,0.576,0.639
3,Moderate,Adaptive Legal RAG,65,0.431,0.815,0.575,0.634
4,Complex,Static Hybrid Legal RAG,75,0.507,0.907,0.648,0.711
5,Complex,Adaptive Legal RAG,75,0.507,0.907,0.648,0.711


Saved: table21_retrieval_by_complexity.csv, table21_retrieval_by_complexity.xlsx
Table 21 ready. n per complexity among 182 answerable.


---
# 9. Experiment 5 — Abstention diagnostics on all 242 questions

This is a retrieval-only abstention diagnostic. It does **not** replace evidence-support verification or legal-judge evaluation. Since the benchmark has no dev split, the threshold sweep should be reported as diagnostic unless a separate calibration set is created.

In [31]:
checks = ['hybrid_retrieve','ROUTE_CONFIG','STATIC_TOP_K','query_embs_all',
          'analyze_question','retrieval_confidence','rrf','bm25_ret','dense_ret',
          'three_sys_details','sva_details','run_static','static_hybrid_retrieve']
for name in checks:
    print(f'{name:25}', 'YES' if name in dir() else '— NO')

hybrid_retrieve           — NO
ROUTE_CONFIG              — NO
STATIC_TOP_K              — NO
query_embs_all            YES
analyze_question          YES
retrieval_confidence      YES
rrf                       YES
bm25_ret                  YES
dense_ret                 YES
three_sys_details         YES
sva_details               — NO
run_static                YES
static_hybrid_retrieve    YES


In [32]:
#@title 9.1 Run Static Hybrid and Adaptive retrieval for the full 242-row benchmark
# Uses the stage-5 functions (static_hybrid_retrieve, adaptive depth map), NOT the old
# hybrid_retrieve/ROUTE_CONFIG. Runs on all 242 (answerable + unanswerable) for abstention.

def static_hybrid_for_all():
    out = []
    for i, row in tqdm(list(bench_all.iterrows()), desc='Static Hybrid full benchmark'):
        out.append(static_hybrid_retrieve(row['question'], query_embs_all[i], k=STATIC_K))
    return out

def adaptive_for_all():
    out, routes = [], []
    for i, row in tqdm(list(bench_all.iterrows()), desc='Adaptive full benchmark'):
        pred = analyze_question(row['question'])
        depth_k = ADAPTIVE_DEPTH.get(str(pred.get('pred_complexity','')).lower(), ADAPTIVE_DEFAULT_K)
        out.append(adaptive_retrieve(row['question'], query_embs_all[i], depth_k))
        routes.append({**pred, 'depth_k': depth_k})
    return out, pd.DataFrame(routes)

static_all_res = static_hybrid_for_all()
adaptive_all_res, adaptive_routes_all = adaptive_for_all()

abst_base = bench_all[['row_id','question_id','question','answerability','complexity',
                       'question_type','legal_domain']].copy()
abst_base['static_confidence']   = [retrieval_confidence(r) for r in static_all_res]
abst_base['adaptive_confidence'] = [retrieval_confidence(r) for r in adaptive_all_res]
abst_base = pd.concat([abst_base.reset_index(drop=True),
                       adaptive_routes_all.add_prefix('adaptive_').reset_index(drop=True)], axis=1)
display(abst_base.head())
save_table(abst_base, 'detail08_full_benchmark_retrieval_confidence')
print(f'Done: {len(bench_all)} questions, confidence scores for Static and Adaptive saved.')

Static Hybrid full benchmark:   0%|          | 0/242 [00:00<?, ?it/s]

Adaptive full benchmark:   0%|          | 0/242 [00:00<?, ?it/s]

,row_id,question_id,question,answerability,complexity,question_type,legal_domain,static_confidence,adaptive_confidence,adaptive_pred_question_type,adaptive_pred_complexity,adaptive_pred_route,adaptive_pred_unanswerable_risk,adaptive_depth_k
0,0,TEST_0210,Жер учаскесін заңсыз пайдаланған жағдайда төле...,answerable,complex,deadline,tax_law,0.043484,0.043484,deadline,moderate,deadline_lookup,0,10
1,1,TEST_0056,KAZAKH INVEST агенттігі инвестиция тарту үшін ...,answerable,moderate,definition,migration_law,0.032522,0.032522,moderate_lookup,moderate,moderate_lookup,0,10
2,2,TEST_0235,Уранды барлауға арналған келісімшарт мерзімін ...,answerable,complex,deadline,civil_law,0.032522,0.032522,deadline,moderate,deadline_lookup,0,10
3,3,TEST_0227,Жер қойнауы учаскесін сенімгерлік басқару шарт...,answerable,complex,deadline,civil_law,0.032266,0.032266,deadline,moderate,deadline_lookup,0,10
4,4,TEST_0237,Жер қойнауын пайдалануға арналған келісімшартт...,answerable,complex,definition,civil_law,0.030769,0.030769,moderate_lookup,moderate,moderate_lookup,0,10


Saved: detail08_full_benchmark_retrieval_confidence.csv, detail08_full_benchmark_retrieval_confidence.xlsx
Done: 242 questions, confidence scores for Static and Adaptive saved.


In [33]:
#@title 9.2 Threshold sweep: row-level and unique-question-level metrics

def abstention_metrics_from_decisions(df, answer_col='pred_answer'):
    # pred_answer=True means system answers; pred_answer=False means abstains.
    ans = df['answerability'].eq('answerable')
    unans = df['answerability'].eq('unanswerable')
    pred_answer = df[answer_col].astype(bool)
    out = {
        'N': len(df),
        'AnswerableN': int(ans.sum()),
        'UnanswerableN': int(unans.sum()),
        'AbstentionAccuracy': round(float(((ans & pred_answer) | (unans & ~pred_answer)).mean()), 4),
        'CorrectAnswerRate': round(float((pred_answer[ans]).mean()), 4) if ans.any() else np.nan,
        'CorrectRefusalRate': round(float((~pred_answer[unans]).mean()), 4) if unans.any() else np.nan,
        'FalseAnswerRate': round(float((pred_answer[unans]).mean()), 4) if unans.any() else np.nan,
        'FalseAbstentionRate': round(float((~pred_answer[ans]).mean()), 4) if ans.any() else np.nan,
    }
    # Balanced decision score gives equal weight to answering answerable questions and refusing unanswerable questions.
    out['BalancedDecisionScore'] = round(float(np.nanmean([out['CorrectAnswerRate'], out['CorrectRefusalRate']])), 4)
    return out


def threshold_sweep(df, score_col, system_name):
    scores = df[score_col].fillna(0).astype(float).to_numpy()
    thresholds = np.unique(np.quantile(scores, np.linspace(0, 1, 101)))
    thresholds = np.unique(np.concatenate([[scores.min() - 1e-9], thresholds, [scores.max() + 1e-9]]))
    rows = []
    for th in thresholds:
        tmp = df.copy()
        tmp['pred_answer'] = tmp[score_col] >= th
        row = {'System': system_name, 'Threshold': float(th), 'Level': 'row'}
        row.update(abstention_metrics_from_decisions(tmp))
        rows.append(row)
        # Unique-question-level: duplicates are collapsed by question text. Confidence is averaged per question.
        uniq = tmp.groupby('question', as_index=False).agg({
            'answerability': 'first',
            score_col: 'mean',
            'pred_answer': 'first',
        })
        row = {'System': system_name, 'Threshold': float(th), 'Level': 'unique_question'}
        row.update(abstention_metrics_from_decisions(uniq))
        rows.append(row)
    return pd.DataFrame(rows)

sweep_static = threshold_sweep(abst_base, 'static_confidence', 'Static Hybrid RAG')
sweep_adaptive = threshold_sweep(abst_base, 'adaptive_confidence', 'Adaptive LegalRAG')
abst_sweep_df = pd.concat([sweep_static, sweep_adaptive], ignore_index=True)
save_table(abst_sweep_df, 'table08_abstention_threshold_sweep')

# Baseline that always answers.
always_answer = abst_base.copy()
always_answer['pred_answer'] = True
no_abst_row = {'System': 'No abstention: always answer', 'Threshold': np.nan, 'Level': 'row'}
no_abst_row.update(abstention_metrics_from_decisions(always_answer))

best_rows = []
for system in abst_sweep_df['System'].unique():
    for level in ['row', 'unique_question']:
        sub = abst_sweep_df[(abst_sweep_df['System'].eq(system)) & (abst_sweep_df['Level'].eq(level))].copy()
        best = sub.sort_values(['BalancedDecisionScore', 'AbstentionAccuracy'], ascending=False).iloc[0].to_dict()
        best['Selection'] = 'diagnostic_best_threshold_on_full_benchmark'
        best_rows.append(best)

abst_best_df = pd.concat([pd.DataFrame([no_abst_row]), pd.DataFrame(best_rows)], ignore_index=True)
display(abst_best_df)
save_table(abst_best_df, 'table08_abstention_summary')

print('Important reporting note: threshold-selected rows are diagnostic because the benchmark has no separate calibration split.')

Saved: table08_abstention_threshold_sweep.csv, table08_abstention_threshold_sweep.xlsx


,System,Threshold,Level,N,AnswerableN,UnanswerableN,AbstentionAccuracy,CorrectAnswerRate,CorrectRefusalRate,FalseAnswerRate,FalseAbstentionRate,BalancedDecisionScore,Selection
0,No abstention: always answer,NaN,row,242,182,60,0.7521,1.0000,0.0,1.0,0.0000,0.5000,NaN
1,Static Hybrid RAG,0.031172,row,242,182,60,0.8678,0.8901,0.8,0.2,0.1099,0.8451,diagnostic_best_threshold_on_full_benchmark
2,Static Hybrid RAG,0.031172,unique_question,186,181,5,0.8871,0.8895,0.8,0.2,0.1105,0.8448,diagnostic_best_threshold_on_full_benchmark
3,Adaptive LegalRAG,0.031172,row,242,182,60,0.8678,0.8901,0.8,0.2,0.1099,0.8451,diagnostic_best_threshold_on_full_benchmark
4,Adaptive LegalRAG,0.031172,unique_question,186,181,5,0.8871,0.8895,0.8,0.2,0.1105,0.8448,diagnostic_best_threshold_on_full_benchmark


Saved: table08_abstention_summary.csv, table08_abstention_summary.xlsx
Important reporting note: threshold-selected rows are diagnostic because the benchmark has no separate calibration split.


---
# 10. Experiment 4 — End-to-End Generation Scaffold

This section creates a reproducible generation input table and optional API-based generation hooks. It is safe by default: `RUN_GENERATION = False`. Turn it on only when you are ready to spend API/GPU resources.

In [35]:
for n in ['static_ans_res','static_res','static_all_res','three_sys_details']:
    print(n, 'YES' if n in dir() else '— NO')

static_ans_res — NO
static_res YES
static_all_res YES
three_sys_details YES


In [36]:
#@title 10.1 Evidence context builder and generation prompts
RUN_GENERATION = False
GENERATION_MODEL = os.environ.get('LEGALRAG_GENERATOR_MODEL', 'gpt-4.1')

def evidence_context_from_result(res, max_chunks=5, max_chars_per_chunk=1200):
    blocks = []
    for rank, (cid, doc, src, title, text) in enumerate(zip(
        res.get('retrieved_ids', []),
        res.get('retrieved_docs', []),
        res.get('retrieved_sources', []),
        res.get('retrieved_titles', []),
        res.get('retrieved_texts', []),
    ), 1):
        if rank > max_chunks:
            break
        t = str(text).replace('\n', ' ').strip()[:max_chars_per_chunk]
        blocks.append(f'[E{rank}] chunk_id={cid}; doc_id={doc}; source={src}; title={title}\n{t}')
    return '\n\n'.join(blocks)

PROMPT_TEMPLATES = {
    'LLM-only': """Answer the Kazakh legal question directly. If you are unsure, say that the answer cannot be determined.\n\nQuestion:\n{question}""",
    'RAG-basic': """Use the evidence to answer the Kazakh legal question.\n\nQuestion:\n{question}\n\nEvidence:\n{evidence}""",
    'RAG-citation': """Answer the Kazakh legal question using only the evidence. Cite the evidence block ids such as [E1].\n\nQuestion:\n{question}\n\nEvidence:\n{evidence}""",
    'RAG-evidence-only': """You are a legal QA system. Answer only if the evidence directly supports the answer. If the evidence is insufficient, refuse with: 'Берілген дереккөздерде бұл сұраққа жеткілікті негіз жоқ.' Include evidence citations [E1], [E2].\n\nQuestion:\n{question}\n\nEvidence:\n{evidence}""",
    'RAG-self-verification': """First draft an answer from the evidence. Then verify whether every legal claim is supported. Return only the final verified answer. If support is insufficient, refuse. Use citations [E1], [E2].\n\nQuestion:\n{question}\n\nEvidence:\n{evidence}""",
    'Static Full RAG': """Answer the Kazakh legal question strictly from the evidence. Requirements: (1) preserve legal conditions and deadlines; (2) cite evidence ids; (3) do not add unsupported legal claims; (4) refuse if evidence is insufficient.\n\nQuestion:\n{question}\n\nEvidence:\n{evidence}""",
}

generation_inputs = []
for pos, (i, row) in enumerate(bench_ans.iterrows()):
    evidence = evidence_context_from_result(static_res[pos], max_chunks=10)
    for variant, template in PROMPT_TEMPLATES.items():
        generation_inputs.append({
            'question_id': row['question_id'],
            'row_id': row['row_id'],
            'variant': variant,
            'question': row['question'],
            'gold_answer': row.get('gold_answer', ''),
            'gold_evidence': row.get('gold_evidence', ''),
            'gold_chunk_id': row.get('gold_chunk_id_str', ''),
            'gold_doc_id': row.get('gold_doc_id_str', ''),
            'gold_source': row.get('gold_source_str', ''),
            'prompt': template.format(question=row['question'], evidence=evidence),
            'retrieved_chunk_ids': '|'.join(str(x) for x in static_res[pos].get('retrieved_ids', [])),
            'retrieved_sources': '|'.join(str(x) for x in static_res[pos].get('retrieved_sources', [])),
        })

generation_inputs_df = pd.DataFrame(generation_inputs)
display(generation_inputs_df.head(3))
save_table(generation_inputs_df, 'generation_inputs_answerable_182')

,question_id,row_id,variant,question,gold_answer,gold_evidence,gold_chunk_id,gold_doc_id,gold_source,prompt,retrieved_chunk_ids,retrieved_sources
0,TEST_0210,0,LLM-only,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,Answer the Kazakh legal question directly. If ...,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...
1,TEST_0210,0,RAG-basic,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,Use the evidence to answer the Kazakh legal qu...,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...
2,TEST_0210,0,RAG-citation,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,Answer the Kazakh legal question using only th...,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...


Saved: generation_inputs_answerable_182.csv, generation_inputs_answerable_182.xlsx


In [37]:
#@title 10.2 Optional generation runner
# This cell is intentionally optional. It will skip unless RUN_GENERATION=True and an API key/provider is configured.

def call_generator(prompt: str, model: str = GENERATION_MODEL) -> str:
    # Minimal OpenAI-compatible implementation. Replace with your local model call if needed.
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.responses.create(model=model, input=prompt, temperature=0)
        return resp.output_text
    except Exception as e:
        return f'GENERATION_SKIPPED_OR_FAILED: {type(e).__name__}: {e}'

if RUN_GENERATION:
    gen_rows = []
    for _, row in tqdm(generation_inputs_df.iterrows(), total=len(generation_inputs_df), desc='Generating answers'):
        answer = call_generator(row['prompt'])
        gen_rows.append({**row.to_dict(), 'generated_answer': answer, 'generator_model': GENERATION_MODEL})
    generated_answers_df = pd.DataFrame(gen_rows)
else:
    generated_answers_df = generation_inputs_df.copy()
    generated_answers_df['generated_answer'] = ''
    generated_answers_df['generator_model'] = GENERATION_MODEL
    generated_answers_df['status'] = 'template_only_RUN_GENERATION_false'

save_table(generated_answers_df, 'generation_outputs_or_template')
display(generated_answers_df.head(3))

Saved: generation_outputs_or_template.csv, generation_outputs_or_template.xlsx


,question_id,row_id,variant,question,gold_answer,gold_evidence,gold_chunk_id,gold_doc_id,gold_source,prompt,retrieved_chunk_ids,retrieved_sources,generated_answer,generator_model,status
0,TEST_0210,0,LLM-only,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,Answer the Kazakh legal question directly. If ...,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...,,gpt-4.1,template_only_RUN_GENERATION_false
1,TEST_0210,0,RAG-basic,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,Use the evidence to answer the Kazakh legal qu...,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...,,gpt-4.1,template_only_RUN_GENERATION_false
2,TEST_0210,0,RAG-citation,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,Answer the Kazakh legal question using only th...,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...,,gpt-4.1,template_only_RUN_GENERATION_false


---
# 10b. Generator backbone comparison (LLM-only vs RAG)

Local 7-8B models run on Colab GPU in 4-bit; GPT-4.1 via OpenAI API. GPT-4.1 LLM-only is included as a **shared anchor** so the model contribution and the retrieval contribution can be separated. Off by default (`RUN_GENERATOR_COMPARISON=False`, `RUN_JUDGE=False`).

In [38]:
#@title 10.3a Extra deps for local generator models (4-bit)
!pip install -q bitsandbytes accelerate sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.1 MB/s eta 0:00:00


In [39]:
#@title 10.3b Generator backbone comparison — configuration
# Compares LLM-only vs RAG-conditioned generation across backbone models.
# Local 7-8B models run on Colab GPU in 4-bit. GPT-4.1 runs via OpenAI API.
import os, gc, time, json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RUN_GENERATOR_COMPARISON = False   # set True to actually generate (uses GPU + API budget)
GEN_MAX_NEW_TOKENS = 512
GEN_EVAL_N = 50                    # questions per system (subset of bench_ans for cost control)

# Each entry: (display_name, hf_or_api_id, backend, setting, language)
#   backend: 'hf' (local transformers 4-bit) or 'openai'
#   setting: 'LLM-only' or 'RAG'
#   language: 'kz' or 'rus'  (controls which prompt language is used)
GEN_SYSTEMS = [
    ('KazLLM-1.0-8B',          'issai/LLama-3.1-KazLLM-1.0-8B',       'hf',     'LLM-only', 'kz'),
    ('Hermes-3-Llama-3.1-8B',  'NousResearch/Hermes-3-Llama-3.1-8B',  'hf',     'LLM-only', 'kz'),
    ('Qwen2.5-7B',             'Qwen/Qwen2.5-7B-Instruct',            'hf',     'LLM-only', 'kz'),
    ('GPT-4.1 (LLM-only)',     'gpt-4.1',                             'openai', 'LLM-only', 'kz'),   # shared anchor
    ('GPT-4.1 (RAG KZ)',       'gpt-4.1',                             'openai', 'RAG',      'kz'),
    ('GPT-4.1 (RAG RUS)',      'gpt-4.1',                             'openai', 'RAG',      'rus'),
]

# Evaluation subset (stratified by complexity for fairness).
_eval_pool = bench_ans.copy()
if 'complexity' in _eval_pool.columns and len(_eval_pool) > GEN_EVAL_N:
    eval_subset = (_eval_pool.groupby('complexity', group_keys=False)
                   .apply(lambda s: s.sample(min(len(s), max(1, GEN_EVAL_N // 3)),
                                             random_state=RANDOM_SEED)))
    eval_subset = eval_subset.head(GEN_EVAL_N).reset_index(drop=True)
else:
    eval_subset = _eval_pool.head(GEN_EVAL_N).reset_index(drop=True)
print(f'Generator comparison eval subset: {len(eval_subset)} questions')

# Map eval_subset rows back to their static retrieval result (for RAG evidence).
_rowid_to_pos = {rid: i for i, rid in enumerate(bench_ans['row_id'].tolist())}
def evidence_for_row(row):
    pos = _rowid_to_pos.get(row['row_id'])
    if pos is None:
        return ''
    return evidence_context_from_result(static_ans_res[pos], max_chunks=8)


Generator comparison eval subset: 48 questions


In [40]:
#@title 10.3c Bilingual prompts for the comparison
PROMPTS_BY_LANG = {
    'kz': {
        'LLM-only': "Қазақ тіліндегі заң сұрағына нақты жауап беріңіз. Егер сенімді болмасаңыз, жауап анықталмайтынын айтыңыз.\n\nСұрақ:\n{question}",
        'RAG': "Берілген дереккөздерді пайдаланып, қазақ тіліндегі заң сұрағына жауап беріңіз. Дереккөз болмаса, бас тартыңыз. Дереккөздерге сілтеме жасаңыз [E1], [E2].\n\nСұрақ:\n{question}\n\nДереккөздер:\n{evidence}",
    },
    'rus': {
        'LLM-only': "Дайте точный ответ на юридический вопрос. Если не уверены, укажите, что ответ не может быть определён.\n\nВопрос:\n{question}",
        'RAG': "Используя приведённые источники, ответьте на юридический вопрос. Если источников недостаточно — откажитесь. Цитируйте источники [E1], [E2].\n\nВопрос:\n{question}\n\nИсточники:\n{evidence}",
    },
}

def build_prompt(setting, language, question, evidence=''):
    tmpl = PROMPTS_BY_LANG[language][setting]
    return tmpl.format(question=question, evidence=evidence)


In [41]:
#@title 10.3d Backends: local 4-bit HF model + OpenAI
_HF_CACHE = {}

def load_hf_4bit(model_id):
    if model_id in _HF_CACHE:
        return _HF_CACHE[model_id]
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map='auto',
        trust_remote_code=True, torch_dtype=torch.float16)
    model.eval()
    _HF_CACHE[model_id] = (tok, model)
    return tok, model

def free_hf(model_id):
    if model_id in _HF_CACHE:
        tok, model = _HF_CACHE.pop(model_id)
        del tok, model
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gen_hf(model_id, prompt, max_new_tokens=GEN_MAX_NEW_TOKENS):
    import torch
    tok, model = load_hf_4bit(model_id)
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        inputs = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                         return_tensors='pt').to(model.device)
    except Exception:
        inputs = tok(prompt, return_tensors='pt').input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=None, top_p=None,
                             pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip()

def gen_openai(model_id, prompt, max_new_tokens=GEN_MAX_NEW_TOKENS):
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model_id, temperature=0, max_tokens=max_new_tokens,
            messages=[{'role': 'user', 'content': prompt}])
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f'GENERATION_FAILED: {type(e).__name__}: {e}'


In [42]:
#@title 10.3e Run generation across all systems
if RUN_GENERATOR_COMPARISON:
    gen_records = []
    for name, model_id, backend, setting, lang in GEN_SYSTEMS:
        print(f'\n=== {name} | {backend} | {setting} | {lang} ===')
        for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc=name):
            evidence = evidence_for_row(row) if setting == 'RAG' else ''
            prompt = build_prompt(setting, lang, row['question'], evidence)
            t0 = time.time()
            if backend == 'hf':
                answer = gen_hf(model_id, prompt)
            else:
                answer = gen_openai(model_id, prompt)
            gen_records.append({
                'system': name, 'model_id': model_id, 'backend': backend,
                'setting': setting, 'language': lang,
                'row_id': row['row_id'], 'question_id': row.get('question_id', ''),
                'question': row['question'],
                'gold_answer': row.get('gold_answer', ''),
                'gold_evidence': row.get('gold_evidence', ''),
                'generated_answer': answer,
                'latency_s': round(time.time() - t0, 2),
            })
        if backend == 'hf':
            free_hf(model_id)   # release GPU before next local model
    gen_compare_df = pd.DataFrame(gen_records)
    save_table(gen_compare_df, 'table14_generator_comparison_raw')
    display(gen_compare_df.groupby(['system', 'setting', 'language']).size()
            .rename('n').reset_index())
else:
    print('RUN_GENERATOR_COMPARISON is False. Set it True to generate (GPU + API budget needed).')
    gen_compare_df = pd.DataFrame()


RUN_GENERATOR_COMPARISON is False. Set it True to generate (GPU + API budget needed).


In [43]:
#@title 10.3f LLM-judge scoring (GPT-4o) + automatic metrics
JUDGE_MODEL = 'gpt-4o'   # pin exact snapshot in the paper (model + access date)
RUN_JUDGE = False        # set True after generation; uses API budget

import re as _re
try:
    import sacrebleu
    def chrf_pp(hyp, ref):
        if not str(ref).strip():
            return np.nan
        return sacrebleu.sentence_chrf(str(hyp), [str(ref)],
                                       word_order=2).score   # chrF++
except Exception:
    def chrf_pp(hyp, ref):
        return np.nan

def token_f1(hyp, ref):
    h = set(str(hyp).lower().split()); r = set(str(ref).lower().split())
    if not h or not r:
        return 0.0
    tp = len(h & r)
    if tp == 0:
        return 0.0
    p = tp / len(h); rc = tp / len(r)
    return 2 * p * rc / (p + rc)

JUDGE_PROMPT = """You are a strict bilingual (Kazakh/Russian) legal QA evaluator.
Compare the SYSTEM ANSWER against the GOLD ANSWER for the QUESTION.
Score each dimension on an integer 0-2 scale (hallucination is 0=none,1=minor,2=major).
Treat Kazakh morphological variants as equivalent. Return ONLY JSON:
{{"legal_correctness":0-2,"evidence_support":0-2,"completeness":0-2,"clarity":0-2,"hallucination":0-2}}

QUESTION:
{question}

GOLD ANSWER:
{gold}

SYSTEM ANSWER:
{answer}
"""

def judge_one(question, gold, answer):
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=JUDGE_MODEL, temperature=0,
            messages=[{'role': 'user',
                       'content': JUDGE_PROMPT.format(question=question, gold=gold, answer=answer)}])
        txt = resp.choices[0].message.content.strip()
        txt = _re.sub(r'```json|```', '', txt).strip()
        return json.loads(txt)
    except Exception as e:
        return {'error': f'{type(e).__name__}: {e}'}

if RUN_JUDGE and len(gen_compare_df):
    judged = []
    for _, r in tqdm(gen_compare_df.iterrows(), total=len(gen_compare_df), desc='LLM-judge'):
        scores = judge_one(r['question'], r['gold_answer'], r['generated_answer'])
        rec = {**r.to_dict()}
        for dim in ['legal_correctness', 'evidence_support', 'completeness', 'clarity', 'hallucination']:
            rec[f'judge_{dim}'] = scores.get(dim, np.nan) if 'error' not in scores else np.nan
        rec['chrF++'] = chrf_pp(r['generated_answer'], r['gold_answer'])
        rec['token_f1'] = token_f1(r['generated_answer'], r['gold_answer'])
        judged.append(rec)
    judged_df = pd.DataFrame(judged)
    save_table(judged_df, 'table14_generator_comparison_judged')

    summary = (judged_df.groupby(['system', 'setting', 'language'])
               .agg(N=('row_id', 'size'),
                    legal_correctness=('judge_legal_correctness', 'mean'),
                    evidence_support=('judge_evidence_support', 'mean'),
                    completeness=('judge_completeness', 'mean'),
                    clarity=('judge_clarity', 'mean'),
                    hallucination=('judge_hallucination', 'mean'),
                    chrFpp=('chrF++', 'mean'),
                    token_f1=('token_f1', 'mean'),
                    latency_s=('latency_s', 'mean'))
               .round(3).reset_index())
    display(summary)
    save_table(summary, 'table14_generator_comparison_summary')
    print('Note for paper: GPT-4.1 LLM-only is the shared anchor — compare it against '
          'GPT-4.1 RAG to isolate the retrieval contribution, and against the local '
          'LLM-only models to isolate the backbone contribution. '
          'Judge model and generator share the GPT family: report self-preference bias as a limitation, '
          'ideally cross-check a subset with a non-GPT judge.')
else:
    print('RUN_JUDGE is False or no generations available.')


RUN_JUDGE is False or no generations available.


In [44]:
#@title 10.3g Table 22 — Generator backbone comparison (6 clean metrics)
# Metrics: Legal correctness, Evidence support, Hallucination (LLM-judge 0-2),
#          chrF++, Correct/False abstention, Latency.
# CitAcc is computed by checking whether a cited [En] marker points at the GOLD evidence chunk.
import re as _re3

if 'judged_df' in dir() and len(judged_df):
    df = judged_df.copy()

    # --- Answerability per row (to split abstention correctly) ---
    rowid_to_ans = dict(zip(bench_all['row_id'], bench_all['answerability'].astype(str)))
    df['answerability'] = df['row_id'].map(rowid_to_ans).fillna('answerable')

    # --- Abstention detection ---
    ABSTAIN_MARKERS = ['не могу', 'недостаточно', 'жоқ', 'анықтау мүмкін емес', 'мүмкін емес',
                       'нет информации', 'cannot', 'insufficient', 'no information', 'unable to']
    def abstained(ans):
        s = str(ans).lower()
        return int(any(m in s for m in ABSTAIN_MARKERS))
    df['abstain'] = df['generated_answer'].apply(abstained)

    # --- TRUE citation accuracy: does a cited [En] point at the gold evidence chunk? ---
    # Evidence in the RAG prompt is numbered E1..En from evidence_for_row (same order).
    # We reconstruct that order and find which index is the gold chunk, then check the citation.
    def gold_index_in_evidence(row_id):
        pos = _rowid_to_pos.get(row_id)
        if pos is None:
            return None
        res = static_ans_res[pos]
        ret_ids = [clean_str(x) for x in res.get('retrieved_ids', [])][:10]  # matches max_chunks=10 used by evidence_context_from_result (cell 10.1)
        gold = {p for p in _re3.split(r'[|;,]\s*',
                clean_str(dict(zip(bench_all['row_id'], bench_all.get('gold_chunk_id_str', bench_all.get('gold_chunk_id','')) )).get(row_id,''))) if p}
        for idx, cid in enumerate(ret_ids, 1):  # 1-based -> [E1]..
            if cid in gold:
                return idx
        return None  # gold not in the provided evidence -> citation can't be correct

    def cit_acc(ans, setting, row_id):
        if setting != 'RAG':
            return np.nan
        gi = gold_index_in_evidence(row_id)
        if gi is None:
            return 0  # gold not shown or not citable -> cannot cite correctly
        cited = set(int(m) for m in _re3.findall(r'\[E(\d+)\]', str(ans)))
        if not cited:
            return 0
        return int(gi in cited)
    df['cit_acc'] = [cit_acc(a, s, r) for a, s, r in
                     zip(df['generated_answer'], df['setting'], df['row_id'])]

    # --- System/Setting display labels matching the paper ---
    SETTING_LABEL = {
        ('KazLLM-1.0-8B', 'LLM-only'): ('KazLLM-1.0-8B', 'LLM-only'),
        ('Hermes-3-Llama-3.1-8B', 'LLM-only'): ('Hermes-3-Llama-3.1-8B', 'LLM-only'),
        ('Qwen2.5-7B', 'LLM-only'): ('Qwen2.5-7B', 'LLM-only'),
        ('GPT-4.1 (RAG KZ)', 'RAG'): ('GPT-4.1', 'Static Hybrid Legal RAG KZ'),
        ('GPT-4.1 (RAG RUS)', 'RAG'): ('GPT-4.1', 'Static Hybrid Legal RAG RUS'),
        ('GPT-4.1 (LLM-only)', 'LLM-only'): ('GPT-4.1', 'LLM-only (anchor)'),
    }

    t22_rows = []
    for (sysname, setting), sub in df.groupby(['system', 'setting']):
        label = SETTING_LABEL.get((sysname, setting), (sysname, setting))
        ans_sub = sub[sub['answerability'] == 'answerable']
        unans_sub = sub[sub['answerability'] == 'unanswerable']
        t22_rows.append({
            'System / Model': label[0],
            'Setting': label[1],
            'Legal':            round(float(sub['judge_legal_correctness'].mean()), 3),
            'Evidence':         round(float(sub['judge_evidence_support'].mean()), 3),
            'Halluc ↓':         round(float(sub['judge_hallucination'].mean()), 3),
            'chrF++':           round(float(np.nanmean(sub['chrF++'])), 2),
            'CitAcc':           (round(float(np.nanmean(sub['cit_acc'])), 3)
                                 if sub['cit_acc'].notna().any() else None),
            'CorrectAbstain↑':  (round(float(unans_sub['abstain'].mean()), 3)
                                 if len(unans_sub) else None),
            'FalseAbstain↓':    (round(float(ans_sub['abstain'].mean()), 3)
                                 if len(ans_sub) else None),
            'Latency':          round(float(sub['latency_s'].mean()), 2),
        })

    ORDER = ['KazLLM-1.0-8B', 'Hermes-3-Llama-3.1-8B', 'Qwen2.5-7B', 'GPT-4.1']
    table22 = pd.DataFrame(t22_rows)
    table22['_o'] = table22['System / Model'].apply(lambda x: ORDER.index(x) if x in ORDER else 99)
    table22 = table22.sort_values(['_o', 'Setting']).drop(columns='_o').reset_index(drop=True)
    display(table22)
    save_table(table22, 'table22_generator_backbone_comparison')
    print('Table 22 ready. Legal/Evidence: higher better (0-2). Halluc & FalseAbstain: lower better. '
          'CorrectAbstain: higher better (unanswerable only). '
          'CitAcc = cited [En] marker points at the gold evidence chunk. '
          'CAUTION: judge=GPT-4o shares family with GPT-4.1 -> report self-preference bias as a limitation.')
else:
    print('Need judged_df from cell 10.3f first (run generation + judge).')


Need judged_df from cell 10.3f first (run generation + judge).


In [45]:
#@title 10.3h Judge validation — GPT-4o vs GPT-5 agreement (kappa + system-ranking Spearman)
# Validates the LLM judge: do two different judges agree per-item (weighted kappa /
# Krippendorff alpha) and do they induce the SAME system ranking (Spearman)?
# Requires gen_compare_df (from 10.3e) with real generated answers.
import re as _rej, json as _json, time as _t
import numpy as np, pandas as pd
from tqdm.auto import tqdm

RUN_JUDGE_VALIDATION = False           # set True; calls BOTH judges on every answer (2x API cost)
JUDGE_MODELS = {'judgeA': 'gpt-4o', 'judgeB': 'gpt-5'}   # pin exact snapshots + access date in paper
JUDGE_DIMS = ['legal_correctness', 'evidence_support', 'completeness', 'clarity', 'hallucination']

_JUDGE_PROMPT = """You are a strict bilingual (Kazakh/Russian) legal QA evaluator.
Compare the SYSTEM ANSWER against the GOLD ANSWER for the QUESTION.
Score each dimension on an integer 0-2 scale (hallucination: 0=none,1=minor,2=major).
Treat Kazakh morphological variants as equivalent. Return ONLY JSON:
{{"legal_correctness":0-2,"evidence_support":0-2,"completeness":0-2,"clarity":0-2,"hallucination":0-2}}

QUESTION:
{question}

GOLD ANSWER:
{gold}

SYSTEM ANSWER:
{answer}
"""

def _judge_call(model_id, question, gold, answer):
    try:
        from openai import OpenAI
        client = OpenAI()
        kwargs = dict(model=model_id,
                      messages=[{'role': 'user',
                                 'content': _JUDGE_PROMPT.format(question=question, gold=gold, answer=answer)}])
        # GPT-5 family may not accept temperature; try with, fall back without.
        try:
            resp = client.chat.completions.create(temperature=0, **kwargs)
        except Exception:
            resp = client.chat.completions.create(**kwargs)
        txt = _rej.sub(r'```json|```', '', resp.choices[0].message.content.strip()).strip()
        return _json.loads(txt)
    except Exception as e:
        return {'_error': f'{type(e).__name__}: {e}'}

if RUN_JUDGE_VALIDATION and 'gen_compare_df' in dir() and len(gen_compare_df):
    recs = []
    for _, r in tqdm(gen_compare_df.iterrows(), total=len(gen_compare_df), desc='Dual-judge'):
        rec = {'system': r['system'], 'setting': r['setting'], 'row_id': r['row_id']}
        for tag, mid in JUDGE_MODELS.items():
            s = _judge_call(mid, r['question'], r['gold_answer'], r['generated_answer'])
            for d in JUDGE_DIMS:
                rec[f'{tag}_{d}'] = s.get(d, np.nan) if '_error' not in s else np.nan
        recs.append(rec)
    dual = pd.DataFrame(recs)
    save_table(dual, 'table_judge_validation_raw')

    # ---- 1) Per-dimension agreement: weighted Cohen kappa + Krippendorff alpha ----
    try:
        from sklearn.metrics import cohen_kappa_score
        have_sk = True
    except Exception:
        have_sk = False

    def krippendorff_alpha_ordinal(a, b):
        # simple 2-rater ordinal alpha
        pairs = [(x, y) for x, y in zip(a, b) if pd.notna(x) and pd.notna(y)]
        if not pairs:
            return np.nan
        vals = [v for p in pairs for v in p]
        n = len(pairs)
        Do = np.mean([(x - y) ** 2 for x, y in pairs])
        gm = np.mean(vals)
        De = np.mean([(v - gm) ** 2 for v in vals]) * 2
        return 1 - Do / De if De else np.nan

    agree_rows = []
    for d in JUDGE_DIMS:
        a = pd.to_numeric(dual[f'judgeA_{d}'], errors='coerce')
        b = pd.to_numeric(dual[f'judgeB_{d}'], errors='coerce')
        mask = a.notna() & b.notna()
        n = int(mask.sum())
        pa = float((a[mask] == b[mask]).mean()) if n else np.nan
        wk = (cohen_kappa_score(a[mask].astype(int), b[mask].astype(int), weights='quadratic')
              if have_sk and n and a[mask].nunique() > 1 and b[mask].nunique() > 1 else np.nan)
        alpha = krippendorff_alpha_ordinal(a[mask].tolist(), b[mask].tolist())
        agree_rows.append({'Dimension': d, 'N': n,
                           'Percent_agreement': round(pa, 3) if pd.notna(pa) else None,
                           'Weighted_kappa': round(wk, 3) if pd.notna(wk) else None,
                           'Krippendorff_alpha': round(alpha, 3) if pd.notna(alpha) else None})
    judge_agreement = pd.DataFrame(agree_rows)
    display(judge_agreement)
    save_table(judge_agreement, 'table_judge_agreement')

    # ---- 2) System-ranking agreement: do both judges rank systems the same? ----
    from scipy.stats import spearmanr
    rankA = dual.groupby('system')[[f'judgeA_{d}' for d in JUDGE_DIMS]].mean().mean(axis=1)
    rankB = dual.groupby('system')[[f'judgeB_{d}' for d in JUDGE_DIMS]].mean().mean(axis=1)
    rank_tbl = pd.DataFrame({'judgeA_mean': rankA, 'judgeB_mean': rankB})
    rank_tbl['rankA'] = rank_tbl['judgeA_mean'].rank(ascending=False)
    rank_tbl['rankB'] = rank_tbl['judgeB_mean'].rank(ascending=False)
    rho, p = spearmanr(rank_tbl['judgeA_mean'], rank_tbl['judgeB_mean'])
    display(rank_tbl.round(3))
    save_table(rank_tbl.reset_index(), 'table_judge_system_ranking')
    print(f'System-ranking Spearman rho = {rho:.3f} (p = {p:.4f}). '
          'High rho => the "which system is best" conclusion is robust to judge choice.')
    print('Report in paper: GPT-5 is the primary judge (Section 3.7.5); GPT-4o cross-check. '
          'State self-preference caveat: GPT-4.1 generations judged by GPT-family judges.')
else:
    print('Set RUN_JUDGE_VALIDATION=True and ensure gen_compare_df has real generated answers.')


Set RUN_JUDGE_VALIDATION=True and ensure gen_compare_df has real generated answers.


---
# 11. Generation / Judge Evaluation Templates

Legal correctness, support rate, citation accuracy, hallucination rate, and unsupported claim rate require either human annotation or a configured evidence-grounded judge. This section creates a clean annotation template.

In [46]:
#@title 11.1 Create judge / human evaluation template
judge_template_cols = [
    'question_id', 'row_id', 'variant', 'question', 'gold_answer', 'gold_evidence',
    'gold_chunk_id', 'gold_doc_id', 'gold_source', 'retrieved_chunk_ids', 'retrieved_sources',
    'generated_answer',
    'legal_correctness_0_1_2', 'answer_completeness_0_1_2', 'evidence_support_0_1_2',
    'citation_accuracy_0_1', 'hallucination_yes_no', 'unsupported_claim_rate', 'notes'
]
for c in judge_template_cols:
    if c not in generated_answers_df.columns:
        generated_answers_df[c] = ''
judge_template_df = generated_answers_df[judge_template_cols].copy()
save_table(judge_template_df, 'generation_judge_annotation_template')
display(judge_template_df.head(3))

Saved: generation_judge_annotation_template.csv, generation_judge_annotation_template.xlsx


,question_id,row_id,variant,question,gold_answer,gold_evidence,gold_chunk_id,gold_doc_id,gold_source,retrieved_chunk_ids,retrieved_sources,generated_answer,legal_correctness_0_1_2,answer_completeness_0_1_2,evidence_support_0_1_2,citation_accuracy_0_1,hallucination_yes_no,unsupported_claim_rate,notes
0,TEST_0210,0,LLM-only,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...,,,,,,,,
1,TEST_0210,0,RAG-basic,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...,,,,,,,,
2,TEST_0210,0,RAG-citation,Жер учаскесін заңсыз пайдаланған жағдайда төле...,Жазбаша нұсқама табыс етілген күннен бастап тө...,бюджетке төленеді.Қызметін арнайы экономикалық...,788,САЛЫҚ ЖӘНЕ БЮДЖЕТКЕ ТӨЛЕНЕТІН БАСҚА ДА МІНДЕТТ...,https://adilet.zan.kz/kaz/docs/K1700000120,0|788|787|298|128|783|5|12|770|229,https://adilet.zan.kz/kaz/docs/V19UB004963|htt...,,,,,,,,


In [49]:
#@title 11.2 Aggregation from adjudicated (final) labels — adjudicated version
COMPLETED_JUDGE_FILE = '/content/adjudicated_evaluation_100.xlsx'

if COMPLETED_JUDGE_FILE and Path(COMPLETED_JUDGE_FILE).exists():
    # adjudicated file: final scores live on the 'Adjudicated' sheet
    try:
        labeled = pd.read_excel(COMPLETED_JUDGE_FILE, sheet_name='Adjudicated')
    except (ValueError, KeyError):
        labeled = pd.read_excel(COMPLETED_JUDGE_FILE)  # fallback: first sheet

    print('Loaded file:', COMPLETED_JUDGE_FILE)
    print('Columns:', list(labeled.columns))

    # Detect grouping column
    if 'variant' in labeled.columns:
        group_col = 'variant'
    elif 'system' in labeled.columns:
        group_col = 'system'
    elif 'System' in labeled.columns:
        group_col = 'System'
    elif 'Variant' in labeled.columns:
        group_col = 'Variant'
    else:
        labeled['_all_rows'] = 'Adjudicated Human Evaluation'
        group_col = '_all_rows'
        print('No variant/system column found. Aggregating all rows together.')

    # Candidate metric columns — adjudicated file uses the _FINAL suffix.
    # Both suffixed and plain names are listed so the cell also works on raw annotator files.
    metric_candidates = [
        'legal_correctness_0_1_2_FINAL',
        'completeness_0_1_2_FINAL',
        'answer_completeness_0_1_2_FINAL',
        'evidence_support_0_1_2_FINAL',
        'citation_accuracy_0_1_FINAL',
        'clarity_0_1_2_FINAL',
        'hallucination_yes_no_FINAL',
        # plain (raw annotator) fallbacks
        'legal_correctness_0_1_2',
        'answer_completeness_0_1_2',
        'completeness_0_1_2',
        'evidence_support_0_1_2',
        'citation_accuracy_0_1',
        'clarity_0_1_2',
        'hallucination_yes_no',
        'notes',
    ]

    existing_metrics = [c for c in metric_candidates if c in labeled.columns]
    print('Detected metric columns:', existing_metrics)

    if not existing_metrics:
        print(
            'No filled judge/human metric columns detected. '
            'This file is probably still an empty annotation template.'
        )
    else:
        agg_rows = []
        for variant, sub in labeled.groupby(group_col, dropna=False):
            row = {'Variant': variant, 'N': len(sub)}
            for col in existing_metrics:
                if col == 'notes':
                    continue
                if col in ['hallucination_yes_no', 'hallucination',
                           'hallucination_yes_no_FINAL']:
                    row['HallucinationRate'] = round(
                        sub[col].astype(str).str.strip().str.lower().isin(
                            ['yes', 'true', '1', 'иә', 'да']
                        ).mean(),
                        4
                    )
                else:
                    row[col] = round(
                        pd.to_numeric(sub[col], errors='coerce').mean(),
                        4
                    )
            agg_rows.append(row)

        generation_quality_df = pd.DataFrame(agg_rows)
        display(generation_quality_df)
        save_table(
            generation_quality_df,
            'table09_adjudicated_human_quality_from_labels'
        )
else:
    print(
        'No completed judge file configured or file does not exist. '
        'Fill / adjudicate the annotations, then rerun this cell.'
    )

Loaded file: /content/adjudicated_evaluation_100.xlsx
Columns: ['row_id', 'question_id', 'question', 'legal_domain', 'gold_answer', 'legal_correctness_0_1_2_A1', 'legal_correctness_0_1_2_A2', 'legal_correctness_0_1_2_FINAL', 'legal_correctness_0_1_2_RULE', 'citation_accuracy_0_1_A1', 'citation_accuracy_0_1_A2', 'citation_accuracy_0_1_FINAL', 'citation_accuracy_0_1_RULE', 'evidence_support_0_1_2_A1', 'evidence_support_0_1_2_A2', 'evidence_support_0_1_2_FINAL', 'evidence_support_0_1_2_RULE', 'completeness_0_1_2_A1', 'completeness_0_1_2_A2', 'completeness_0_1_2_FINAL', 'completeness_0_1_2_RULE', 'clarity_0_1_2_A1', 'clarity_0_1_2_A2', 'clarity_0_1_2_FINAL', 'clarity_0_1_2_RULE', 'hallucination_yes_no_A1', 'hallucination_yes_no_A2', 'hallucination_yes_no_FINAL', 'hallucination_yes_no_RULE', 'n_disagreements_row']
No variant/system column found. Aggregating all rows together.
Detected metric columns: ['legal_correctness_0_1_2_FINAL', 'completeness_0_1_2_FINAL', 'evidence_support_0_1_2_FINAL

,Variant,N,legal_correctness_0_1_2_FINAL,completeness_0_1_2_FINAL,evidence_support_0_1_2_FINAL,citation_accuracy_0_1_FINAL,clarity_0_1_2_FINAL,HallucinationRate
0,Adjudicated Human Evaluation,100,1.79,1.76,1.83,0.93,1.85,0.03


Saved: table09_adjudicated_human_quality_from_labels.csv, table09_adjudicated_human_quality_from_labels.xlsx


In [50]:
#@title 11.3 Inter-annotator agreement (pre-adjudication) for the paper
# Reads the Agreement_pre_adjud sheet from the adjudicated workbook.
AGREEMENT_SHEET = 'Agreement_pre_adjud'
try:
    agree = pd.read_excel(COMPLETED_JUDGE_FILE, sheet_name=AGREEMENT_SHEET, header=2)
    agree = agree.dropna(how='all')
    display(agree)
    save_table(agree, 'table10_inter_annotator_agreement')
    print('Reminder: agreement is computed BEFORE adjudication; '
          'final quality scores in 11.2 are computed AFTER adjudication.')
    print('Note for the paper: low/negative kappa with high % agreement reflects the '
          'kappa paradox (one annotator used near-constant max ratings -> low label variance), '
          'not unreliable annotation. Report both % agreement and kappa, and state this explicitly.')
except Exception as e:
    print('Could not load agreement sheet (is adjudicated_evaluation_100.xlsx present?):', e)


,Criterion,N pairs (both rated),% agreement,Cohen kappa,Weighted kappa,Interpretation
0,legal_correctness_0_1_2,100.0,79.0,-0.076,-0.080,Worse than chance
1,citation_accuracy_0_1,80.0,92.5,0.233,NaN,Fair
2,evidence_support_0_1_2,80.0,80.0,0.052,-0.008,Slight
3,completeness_0_1_2,100.0,79.0,0.137,0.058,Slight
4,clarity_0_1_2,100.0,85.0,-0.019,-0.019,Worse than chance
5,hallucination_yes_no,100.0,97.0,-0.014,NaN,Worse than chance
7,Note: pairs where one annotator left the cell ...,NaN,NaN,NaN,NaN,NaN
8,Note: kappa is low/negative on several criteri...,NaN,NaN,NaN,NaN,NaN
9,"(very low label variance), which deflates chan...",NaN,NaN,NaN,NaN,NaN


Saved: table10_inter_annotator_agreement.csv, table10_inter_annotator_agreement.xlsx
Reminder: agreement is computed BEFORE adjudication; final quality scores in 11.2 are computed AFTER adjudication.
Note for the paper: low/negative kappa with high % agreement reflects the kappa paradox (one annotator used near-constant max ratings -> low label variance), not unreliable annotation. Report both % agreement and kappa, and state this explicitly.


In [51]:
#@title 11.4 Qualitative QA examples (generated answer + retrieved evidence) for the appendix
# Works whether or not generation was run. If RUN_GENERATION was False, the
# generated answer column will be empty and the table falls back to gold + retrieved.

GEN_SOURCE_CANDIDATES = [
    'generated_answers_df',          # in-memory (cell 10.2)
]
gen_df = None
for name in GEN_SOURCE_CANDIDATES:
    if name in dir() and isinstance(eval(name), pd.DataFrame):
        gen_df = eval(name).copy()
        print('Using in-memory', name, gen_df.shape)
        break
if gen_df is None:
    p = OUTPUT_DIR / 'generation_outputs_or_template.csv'
    if p.exists():
        gen_df = pd.read_csv(p); print('Loaded', p.name, gen_df.shape)
    else:
        gen_df = bench_ans.copy(); print('Fallback to benchmark gold answers.')

def _short(txt, n=500):
    s = '' if txt is None else str(txt).strip()
    s = ' '.join(s.split())
    return s if len(s) <= n else s[:n].rstrip() + ' [...]'

def _col(df, names):
    for c in names:
        if c in df.columns:
            return c
    return None

q_col   = _col(gen_df, ['question'])
ga_col  = _col(gen_df, ['generated_answer', 'rag_answer', 'model_answer'])
gold_col= _col(gen_df, ['gold_answer'])
ev_col  = _col(gen_df, ['gold_evidence', 'evidence'])
rid_col = _col(gen_df, ['question_id', 'row_id'])
dom_col = _col(gen_df, ['legal_domain'])
ans_col = _col(gen_df, ['answerability'])
var_col = _col(gen_df, ['variant', 'System', 'system'])
ret_col = _col(gen_df, ['retrieved_chunk_ids', 'retrieved_ids', 'retrieved_sources'])

# Stratified pick: 2 simple answerable, 2 complex answerable, 2 unanswerable (if columns exist).
pool = gen_df.copy()
picks = []
if ans_col and 'complexity' in pool.columns:
    plan = [
        ((pool[ans_col].eq('answerable')) & (pool['complexity'].eq('simple')),  2),
        ((pool[ans_col].eq('answerable')) & (pool['complexity'].eq('complex')), 2),
        ((pool[ans_col].eq('unanswerable')),                                    2),
    ]
    for mask, k in plan:
        sub = pool[mask]
        if len(sub):
            picks.append(sub.sample(n=min(k, len(sub)), random_state=RANDOM_SEED))
examples = (pd.concat(picks, ignore_index=True) if picks
            else pool.sample(n=min(6, len(pool)), random_state=RANDOM_SEED))

rows = []
for _, r in examples.iterrows():
    rows.append({
        'ID': r.get(rid_col, '') if rid_col else '',
        'Domain': r.get(dom_col, '') if dom_col else '',
        'Answerability': r.get(ans_col, '') if ans_col else '',
        'System': r.get(var_col, '') if var_col else '',
        'Question': _short(r.get(q_col, ''), 300) if q_col else '',
        'Generated answer': _short(r.get(ga_col, ''), 600) if ga_col else '(generation not run)',
        'Gold answer': _short(r.get(gold_col, ''), 400) if gold_col else '',
        'Retrieved evidence': _short(r.get(ev_col, ''), 600) if ev_col else '',
        'Retrieved chunks': _short(r.get(ret_col, ''), 200) if ret_col else '',
    })
examples_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 200)
display(examples_df)
save_table(examples_df, 'table13_qualitative_qa_examples')

# Markdown version for LaTeX appendix
md = ['# Qualitative QA examples\n']
for i, r in examples_df.iterrows():
    md += [f'## Example {i+1} - {r["Domain"]} ({r["Answerability"]})',
           f'**Question:** {r["Question"]}\n',
           f'**Generated answer:** {r["Generated answer"]}\n',
           f'**Gold answer:** {r["Gold answer"]}\n',
           f'**Retrieved evidence:** {r["Retrieved evidence"]}\n', '---\n']
(OUTPUT_DIR / 'table13_qualitative_qa_examples.md').write_text('\n'.join(md), encoding='utf-8')
print('Saved table13_qualitative_qa_examples.{csv,xlsx,md}')


Using in-memory generated_answers_df (1092, 22)


,ID,Domain,Answerability,System,Question,Generated answer,Gold answer,Retrieved evidence,Retrieved chunks
0,TEST_0111,,,RAG-citation,Теміржол вокзалдарында мүгедектерге қандай қызметтер қамтамасыз етілуі тиіс?,,"Теміржол вокзалдарында мүгедектерге арнайы жол белгілері, ыңғайлы қозғалыс жолдары, ақпараттық сигналдық құрылғылар, кезекші кресло-арба, арнайы кабиналар мен таксофондар қамтамасыз етілуі тиіс.",жол осiнен қашықтық бойынша ұлғаюы жағына қарай 30 миллиметрге дейiн және азаю жағына қарай 25 миллиметрге дейiн.11Уәкілетті адам (вокзал қызметкері) жол жүру құжатының (билеттің) қолданылу мерзім...,126|65|64|2|4|127|1|125|63|66
1,TEST_0158,,,RAG-basic,Кен іздеушілікке арналған лицензия қанша уақытқа беріледі және оны ұзарту мүмкін бе?,,Кен іздеушілікке арналған лицензия үш жыл мерзімге беріледі және оны бір рет үш жылға ұзартуға болады.,Кен іздеушілікке арналған лицензия үш жыл мерзімге беріледі. Лицензияны иеленушінің өтініші бойынша көрсетілген мерзім бір рет үш жылға ұзартуға жатады.Егер өтінішті қарау күніне кен іздеушілік уч...,380|379|377|9|378|384|385|381|382|145
2,TEST_0082,,,RAG-self-verification,Қосылған құн салығының асып кетуін қайтару мерзімі қандай?,,"Нөлдік мөлшерлеме бойынша салық салынатын өткізу бойынша айналымдарды жүзеге асыратын салық төлеушіге – елу бес жұмыс күні ішінде, қалған жағдайларда – жетпіс бес жұмыс күні ішінде қайтарылады.","Қосылған құн салығының асып кетуін салық төлеушіге қайтару:1) егер осы Кодекстің 432 және 434-баптарында өзгеше белгіленбесе, осы бапта белгіленген тәртіппен және мерзімдерде;2) салықтық кезеңдегі...",486|647|482|33|481|644|898|488|642|160
3,TEST_0071,,,RAG-self-verification,Жер қойнауын пайдалануды шектеу немесе тыйым салу қандай жағдайларда мүмкін?,,"Ұлттық қауіпсіздікті, халықтың қауіпсіздігін және қоршаған ортаны қорғауды қамтамасыз ету мақсатында жер қойнауын пайдалану шектелуі немесе тыйым салынуы мүмкін.",13-бапқа өзгеріс енгізілді - ҚР 2014Заңымен (алғашқы ресми жарияланған күнінен кейiн күнтiзбелiк он күн өткен соң қолданысқа енгiзiледi).14-бап. Жер қойнауын пайдалануды шектеу және тыйым салу1. Ұ...,460|506|19|96|385|606|53|22|224|97
4,TEST_0193,,,RAG-self-verification,ТҰК және ірі компаниялармен құрылған бірлескен кәсіпорындар санының есебі қай уақытқа дейін жүргізілуі керек?,,Есепті жылдан кейінгі жылдың 1 наурызына қарай.,"""Экономиканың басым секторларына тартылған ТҰК, ірі компаниялар және ""зәкірлік инвесторлармен"" құрылған бірлескен кәсіпорындардың саны"" көрсеткішіКезеңділігі және қалыптастыру мерзімдеріЖыл сайын,...",56|62|6|86|38|28|43|0|7|1
5,TEST_0222,,,LLM-only,Жер қойнауын пайдаланушы салық декларациясын қашан тапсыруы керек?,,"Егер төлем сомасы 10 000 АЕК-тен кем болса, пайдалы қазбаларды өндіруге кіріскен жылынан кейінгі жылдың 31 наурызынан кешіктірмей тапсырады. Егер төлем сомасы 10 000 АЕК-тен асса, тоқсан сайын, ес...",734-бапқа өзгеріс енгізілді – ҚР 2020(2021 бастап қолданысқа енгізіледі) Заңымен.735-бап. Салық декларациясы1. Егер келісімшарт аумағын (жер қойнауы учаскесін) геологиялық зерделеуге және кен орын...,18|585|559|976|588|1001|997|977|611|42


Saved: table13_qualitative_qa_examples.csv, table13_qualitative_qa_examples.xlsx
Saved table13_qualitative_qa_examples.{csv,xlsx,md}


---
# 12. Human Expert Evaluation sample: 100-question stratified subset

In [ ]:
#@title 12.1 Build 100-question human evaluation subset (ALREADY DONE — kept for reproducibility)
REBUILD_HUMAN_SUBSET = False  # subset already annotated & adjudicated; do NOT regenerate

if REBUILD_HUMAN_SUBSET:
    strata = [
        ('simple_answerable',   bench_ans[bench_ans['complexity'].eq('simple')],   20),
        ('moderate_answerable', bench_ans[bench_ans['complexity'].eq('moderate')], 30),
        ('complex_answerable',  bench_ans[bench_ans['complexity'].eq('complex')],  30),
        ('unanswerable',        bench_unans,                                       20),
    ]
    sample_parts = []
    for name, frame, n in strata:
        take = min(n, len(frame))
        sampled = frame.sample(n=take, random_state=RANDOM_SEED).copy() if take else frame.copy()
        sampled['human_eval_stratum'] = name
        sample_parts.append(sampled)
    human_subset = pd.concat(sample_parts, ignore_index=True)
    human_subset = human_subset[[
        'human_eval_stratum', 'row_id', 'question_id', 'question', 'answerability', 'complexity',
        'question_type', 'legal_domain', 'gold_answer', 'gold_evidence', 'gold_chunk_id_str',
        'gold_doc_id_str', 'gold_source_str'
    ]].copy()
    for col in ['legal_correctness_0_1_2', 'citation_accuracy_0_1', 'evidence_support_0_1_2',
                'completeness_0_1_2', 'clarity_0_1_2', 'hallucination_yes_no', 'expert_notes']:
        human_subset[col] = ''
    display(human_subset['human_eval_stratum'].value_counts()
            .rename_axis('stratum').reset_index(name='count'))
    save_table(human_subset, 'human_expert_evaluation_100_template')
    print('Template regenerated. WARNING: re-annotation required — row_ids may differ from existing annotations.')
else:
    print('Skipping subset rebuild — using already-adjudicated 100-question set '
          '(adjudicated_evaluation_100.xlsx).')

Skipping subset rebuild — using already-adjudicated 100-question set (adjudicated_evaluation_100.xlsx).


---
# 12b. Statistical reporting — Bootstrap 95% CI and cross-iteration reproducibility


In [52]:
#@title 12b.1 Bootstrap 95% CI per system (resampling per-question scores)
import numpy as np
import pandas as pd

BOOTSTRAP_N = 1000
CI_LOW, CI_HIGH = 2.5, 97.5
_rng = np.random.default_rng(RANDOM_SEED)
CI_METRICS = ['Hit@1', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10']

def bootstrap_ci(values, n=BOOTSTRAP_N):
    """95% percentile bootstrap CI for the mean of per-question metric values."""
    v = np.asarray(pd.to_numeric(pd.Series(values), errors='coerce').dropna(), dtype=float)
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    means = v[_rng.integers(0, len(v), size=(n, len(v)))].mean(axis=1)
    return float(v.mean()), float(np.percentile(means, CI_LOW)), float(np.percentile(means, CI_HIGH))

# Collect per-question detail frames produced earlier in the notebook.
# retrieval_details (cell 4.3) + sva_details (cell 8.2) cover all reported systems.
SYSTEM_DETAILS = {}
if 'retrieval_details' in dir():
    for sysname, det in retrieval_details.items():
        SYSTEM_DETAILS[sysname] = det
if 'sva_details' in dir():
    for det in sva_details:
        if 'System' in det.columns:
            SYSTEM_DETAILS[str(det['System'].iloc[0])] = det

ci_rows = []
for system, det in SYSTEM_DETAILS.items():
    for m in CI_METRICS:
        if m not in det.columns:
            continue
        mean, lo, hi = bootstrap_ci(det[m])
        ci_rows.append({
            'System': system, 'Metric': m, 'N': int(pd.to_numeric(det[m], errors='coerce').notna().sum()),
            'Mean': round(mean, 4), 'CI95_low': round(lo, 4), 'CI95_high': round(hi, 4),
            'CI_str': f'{mean:.3f} [{lo:.3f}, {hi:.3f}]',
        })

ci_df = pd.DataFrame(ci_rows)
display(ci_df)
save_table(ci_df, 'table11_bootstrap_ci_long')

if len(ci_df):
    ci_wide = ci_df.pivot(index='System', columns='Metric', values='CI_str').reset_index()
    display(ci_wide)
    save_table(ci_wide, 'table11_bootstrap_ci_wide')

print(f'Bootstrap CI: {BOOTSTRAP_N} resamples of per-question scores, 95% percentile interval.')


,System,Metric,N,Mean,CI95_low,CI95_high,CI_str
0,Naive RAG: BM25 k=5,Hit@1,182,0.4176,0.3462,0.4890,"0.418 [0.346, 0.489]"
1,Naive RAG: BM25 k=5,Recall@5,182,0.7692,0.7033,0.8297,"0.769 [0.703, 0.830]"
2,Naive RAG: BM25 k=5,Recall@10,182,0.7692,0.7087,0.8297,"0.769 [0.709, 0.830]"
3,Naive RAG: BM25 k=5,MRR@10,182,0.5500,0.4892,0.6101,"0.550 [0.489, 0.610]"
4,Naive RAG: BM25 k=5,nDCG@10,182,0.6068,0.5454,0.6637,"0.607 [0.545, 0.664]"
5,BM25 k=10,Hit@1,182,0.4176,0.3462,0.4890,"0.418 [0.346, 0.489]"
6,BM25 k=10,Recall@5,182,0.7692,0.7088,0.8297,"0.769 [0.709, 0.830]"
7,BM25 k=10,Recall@10,182,0.8352,0.7802,0.8848,"0.835 [0.780, 0.885]"
8,BM25 k=10,MRR@10,182,0.5584,0.4975,0.6151,"0.558 [0.497, 0.615]"
9,BM25 k=10,nDCG@10,182,0.6278,0.5743,0.6835,"0.628 [0.574, 0.684]"


Saved: table11_bootstrap_ci_long.csv, table11_bootstrap_ci_long.xlsx


Metric,System,Hit@1,MRR@10,Recall@10,Recall@5,nDCG@10
0,BM25 k=10,"0.418 [0.346, 0.489]","0.558 [0.497, 0.615]","0.835 [0.780, 0.885]","0.769 [0.709, 0.830]","0.628 [0.574, 0.684]"
1,Dense BGE-M3 k=10,"0.412 [0.346, 0.484]","0.532 [0.471, 0.595]","0.780 [0.725, 0.841]","0.703 [0.643, 0.758]","0.598 [0.543, 0.657]"
2,Naive RAG: BM25 k=5,"0.418 [0.346, 0.489]","0.550 [0.489, 0.610]","0.769 [0.709, 0.830]","0.769 [0.703, 0.830]","0.607 [0.545, 0.664]"
3,Static Hybrid RAG,"0.451 [0.379, 0.517]","0.594 [0.541, 0.652]","0.857 [0.802, 0.907]","0.791 [0.731, 0.846]","0.658 [0.607, 0.711]"


Saved: table11_bootstrap_ci_wide.csv, table11_bootstrap_ci_wide.xlsx
Bootstrap CI: 1000 resamples of per-question scores, 95% percentile interval.


In [53]:
#@title 12b.2 Cross-iteration reproducibility (run notebook twice: LEGALRAG_ITERATION=1 then =2)
# Each run saves /content/retrieval_iter{ITERATION}.csv in cell 4.3.
# Reported as mean and min-max range over 2 deterministic runs (memory-config variants),
# NOT as a statistical CI (n=2).
from pathlib import Path

iter_files = {1: '/content/retrieval_iter1.csv', 2: '/content/retrieval_iter2.csv'}
frames = [pd.read_csv(p) for p in iter_files.values() if Path(p).exists()]

if len(frames) >= 2:
    runs = pd.concat(frames, ignore_index=True)
    repro_rows = []
    for system, sub in runs.groupby('System'):
        for m in CI_METRICS:
            if m not in sub.columns:
                continue
            vals = pd.to_numeric(sub[m], errors='coerce').dropna().to_numpy()
            if len(vals) == 0:
                continue
            repro_rows.append({
                'System': system, 'Metric': m, 'Runs': len(vals),
                'Mean_runs': round(float(vals.mean()), 4),
                'Min': round(float(vals.min()), 4),
                'Max': round(float(vals.max()), 4),
                'AbsRange': round(float(vals.max() - vals.min()), 4),
            })
    repro_df = pd.DataFrame(repro_rows)
    display(repro_df)
    save_table(repro_df, 'table12_cross_iteration_reproducibility')
    print('Reported as mean and min-max range over deterministic runs. NOT a statistical CI (n=2).')
else:
    print('Only one iteration CSV found. Re-run the whole notebook with '
          "os.environ['LEGALRAG_ITERATION']='2' to produce the second run, then run this cell.")


Only one iteration CSV found. Re-run the whole notebook with os.environ['LEGALRAG_ITERATION']='2' to produce the second run, then run this cell.


---
# 13. Figures and export

In [55]:
for n in ['retrieval_df','three_sys_df','sva_df','table21','abst_best_df','plt']:
    print(n, 'YES' if n in dir() else '— NO')

retrieval_df YES
three_sys_df YES
sva_df — NO
table21 YES
abst_best_df YES
plt YES


In [56]:
#@title 13.1 Figures for paper tables
import matplotlib.pyplot as plt

# --- Fig 1: Retrieval baseline comparison (Naive / BM25 / Dense / Static Hybrid) ---
# source: the three-systems overall table from stage 5 (8.2)
base_df = three_sys_df if 'three_sys_df' in dir() else (
    pd.read_csv(OUTPUT_DIR / 'table_three_systems_overall.csv') if (OUTPUT_DIR / 'table_three_systems_overall.csv').exists()
    else None)

if base_df is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    base_df.set_index('System')[['Hit@1', 'Recall@10', 'MRR@10', 'nDCG@10']].plot(kind='bar', ax=ax)
    ax.set_title('Retrieval: Naive / Static Hybrid / Adaptive (182 answerable)')
    ax.set_ylabel('Score'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig01_three_systems.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('three_sys_df not found — run stage-5 cell 8.2 first.')

# --- Fig 2: Static vs Adaptive by complexity (Table 21) ---
if 'table21' in dir():
    fig, ax = plt.subplots(figsize=(9, 5))
    piv = table21.pivot(index='Complexity Level', columns='System', values='R@1')
    piv.plot(kind='bar', ax=ax)
    ax.set_title('R@1 by Query Complexity — Static vs Adaptive')
    ax.set_ylabel('R@1'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig02_complexity_static_vs_adaptive.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('table21 not found — run cell 8.3 first.')

# --- Fig 3: Abstention diagnostics (if computed) ---
if 'abst_best_df' in dir() and len(abst_best_df):
    plot_abst = abst_best_df[abst_best_df['Level'].eq('row')].copy() if 'Level' in abst_best_df.columns else abst_best_df.copy()
    if len(plot_abst):
        fig, ax = plt.subplots(figsize=(9, 5))
        cols = [c for c in ['CorrectAnswerRate','CorrectRefusalRate','FalseAnswerRate','FalseAbstentionRate'] if c in plot_abst.columns]
        plot_abst.set_index('System')[cols].plot(kind='bar', ax=ax)
        ax.set_title('Abstention Diagnostics')
        ax.set_ylabel('Rate'); ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=20)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'fig03_abstention.png', dpi=180, bbox_inches='tight')
        plt.show()
else:
    print('abst_best_df not found — abstention not computed yet (skip this figure).')

print('Saved figures under', OUTPUT_DIR)

Saved figures under /content/legalrag_outputs


In [57]:
#@title 13.2 Export all outputs as ZIP
zip_path = '/content/legalrag_outputs_dataset_eval.zip'
if Path(zip_path).exists():
    Path(zip_path).unlink()
shutil.make_archive('/content/legalrag_outputs_dataset_eval', 'zip', OUTPUT_DIR)
print(f'ZIP created: {zip_path}')
with zipfile.ZipFile(zip_path) as zf:
    for name in sorted(zf.namelist()):
        print(' -', name)

try:
    from google.colab import files as cf
    cf.download(zip_path)
except Exception:
    print('Not running in Colab; download the ZIP from:', zip_path)

ZIP created: /content/legalrag_outputs_dataset_eval.zip
 - detail01_dense_BGE-M3.csv
 - detail01_dense_BGE-M3.xlsx
 - detail01_dense_E5-Large.csv
 - detail01_dense_E5-Large.xlsx
 - detail01_dense_LaBSE.csv
 - detail01_dense_LaBSE.xlsx
 - detail01_dense_Qwen3-0.6B.csv
 - detail01_dense_Qwen3-0.6B.xlsx
 - detail02_retrieval_baselines.csv
 - detail02_retrieval_baselines.xlsx
 - detail03_text_normalization_or_folding.csv
 - detail03_text_normalization_or_folding.xlsx
 - detail04_reranking_ablation.csv
 - detail04_reranking_ablation.xlsx
 - detail05_query_analyzer_predictions.csv
 - detail05_query_analyzer_predictions.xlsx
 - detail08_full_benchmark_retrieval_confidence.csv
 - detail08_full_benchmark_retrieval_confidence.xlsx
 - detail_adaptive_routes.csv
 - detail_adaptive_routes.xlsx
 - detail_three_systems.csv
 - detail_three_systems.xlsx
 - fig00_answerability.png
 - fig00_complexity.png
 - fig00_legal_domain.png
 - fig00_question_type.png
 - fig01_retrieval_baselines.png
 - fig01_three

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>